# FT-02 : QLoRA — Fine-Tuning avec Quantization
**Navigation** : [Index](README.md) | [<< 01 — Introduction](FT-01-Introduction-FineTuning-Python.ipynb) | [03 — SFT >>](FT-03-Supervised-FineTuning-SFT-Python.ipynb)

**Version anglaise** : [FT-02 — QLoRA (en)](FT-02-QLoRA-Quantization-Python_en.ipynb)

**Objectif** : Comprendre la quantization 4-bit (NF4) et l'approche QLoRA qui combine quantization + LoRA pour fine-tuner des modèles jusqu'a 7B params sur un GPU consumer.

**Plan** :
1. Le problème memoire des grands modèles
2. Quantization : de FP16 a NF4
3. BitsAndBytesConfig en pratique
4. La mécanique LoRA que l'API enveloppe
5. QLoRA : quantized base + LoRA adapters
6. Entraînement QLoRA sur OPT-1.3B
7. Generation avec le modèle QLoRA
8. Comparaison LoRA vs QLoRA
9. Nettoyage memoire GPU
10. Exercices

**Prerequis** : FT-01 (Introduction au Fine-Tuning) — LoRA, rang, adaptateurs.

**Materiel requis** : GPU avec ~8 Go VRAM (QLoRA OPT-1.3B 4-bit = ~2,7 Go).


**Lecture de la sortie committée** : la cellule suivante imprime l'environnement complet (version PyTorch, GPU, VRAM) -- sa sortie committee est la source unique, la prose ne la re-epinglera plus (#9434). L'environnement d'exécution (carte, VRAM totale et libre au démarrage) se lit dans la sortie committée de la cellule suivante -- qui reste la source unique. Le paradoxe pédagogique de ce notebook : cette carte embarquerait OPT-1.3B en FP16 sans aucune difficulté (2,84 Go d'empreinte mesurés plus loin, section 10). Ce n'est pas elle qui justifie QLoRA — c'est la généralisation : la même recette qui tient un 1,3 Md dans ~1 Go de poids quantifiés en fera tenir un 7 Md dans ~6 Go, sur une carte à 300 €. Le notebook déroule la chaîne complète : quantization NF4 (section 2), la mécanique LoRA écrite à la main (section 4), greffe LoRA via l'API (section 5), entraînement réel mesuré (section 6), génération avant/après (section 7), et comparatif mémoire FP16 vs 4-bit (section 8).

In [1]:
import gc
import os
import time
import warnings

# Le chargement d'un checkpoint .bin provoque, par defaut, un thread
# d'arriere-plan de transformers (Thread-auto_conversion) qui interroge
# l'API du Hub pour proposer une conversion safetensors. Sans egress, chaque
# essai imprime un traceback httpx dans les sorties du carnet. La conversion
# est hors objet ici (la source du modele est en lecture seule) : on
# desactive le thread a la source, avant tout chargement.
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"

import torch

warnings.filterwarnings(
    "ignore",
    message=r"_check_is_size will be removed.*",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message=r"torch\.utils\.checkpoint: the use_reentrant parameter.*",
    category=UserWarning,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    vram_free = torch.cuda.mem_get_info()[0] / 1e9
    print(f"VRAM totale : {vram_total:.1f} Go | Libre : {vram_free:.1f} Go")

PyTorch 2.9.0+cu126
CUDA : True
GPU : NVIDIA GeForce RTX 3070 Ti Laptop GPU
VRAM totale : 8.6 Go | Libre : 7.5 Go


### Les paramètres du carnet

Toute la configuration du carnet tient dans la cellule ci-dessous : le taux d'apprentissage, le nombre d'époques, le rang et le rapport alpha/r de l'adaptateur, les modules cibles, la graine. Le reste du carnet ne lit que `CONFIG` — jamais une constante écrite en dur dans une cellule.

C'est ce qui rend le carnet rejouable : changer une valeur et relancer suffit à obtenir une autre mesure, sans toucher au code. Et c'est cette même configuration que le balayage multi-graines de la section 6 fait varier sur trois graines.

In [2]:
# Cellule de PARAMETRES (papermill) : la configuration unique du carnet.
# Tout ce qui pilote l'entrainement passe par elle ; le reste du carnet ne lit
# que CONFIG, jamais une constante ecrite en dur. Rejouer le carnet avec
# d'autres hyperparametres ne demande donc aucune edition de code.
CONFIG = {
    "learning_rate": 2e-4,
    "num_train_epochs": 15,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj"],
    "seed": 42,
    "batch_size": 2,
    "max_length": 256,
}
CONFIG

{'learning_rate': 0.0002,
 'num_train_epochs': 15,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'target_modules': ['q_proj', 'v_proj'],
 'seed': 42,
 'batch_size': 2,
 'max_length': 256}

## 1. Le problème memoire des grands modèles

Fine-tuner un modèle de langue necessite de stocker en VRAM :

| Composant | Taille approximative |
|-----------|---------------------|
| Poids du modèle (FP16) | 2 bytes/param |
| Gradients | 2 bytes/param |
| Etats de l'optimiseur (AdamW) | 8 bytes/param (m + v) |
| Activations (forward pass) | Variable, souvent 1-2x modèle |

**Total pour full fine-tuning** : ~12-16 bytes par paramètre.

| Modèle | Params | Full FT (FP16) | LoRA (FP16) | QLoRA (4-bit) |
|--------|--------|---------------|-------------|---------------|
| GPT-2 | 124M | ~2 Go | ~0.5 Go | ~0.3 Go |
| OPT-1.3B | 1.3B | ~20 Go | ~5 Go | ~2 Go |
| Llama-7B | 7B | ~112 Go | ~28 Go | ~6 Go |

LoRA reduit les gradients et l'optimiseur (seuls les adaptateurs sont entraines), mais les **poids du modèle restent en FP16**.

**QLoRA** va plus loin : il **quantize les poids du modèle en 4-bit** tout en gardant les calculs en precision superieure.


**Le chiffrage réel se lit dans la section 8** : le chargement FP16 d'OPT-1.3B dépasse la théorie des poids seuls, car les tampons CUDA et le contexte résiduel comptent aussi. Mais charger n'est pas fine-tuner : un full fine-tuning exige en plus les gradients et les états de l'optimiseur Adam. C'est cette triple taxe que LoRA (gradients sur les seuls adaptateurs) puis QLoRA (poids de base en 4 bits) réduisent. Les valeurs observées restent dans les outputs afin que la prose ne fige pas une mesure propre à une machine.

## 2. Quantization : de FP16 a NF4

### 2a. Principe de la quantization

La quantization reduit la precision des poids pour diminuer l'empreinte memoire :

```
FP32 (32 bits) → FP16 (16 bits) → INT8 (8 bits) → NF4 (4 bits)
4 bytes/param    2 bytes/param    1 byte/param    0.5 bytes/param
```

### 2b. Pourquoi NF4 et pas un simple INT4 ?

Les poids des réseaux de neurones suivent une distribution **approximativement normale** (centree sur 0). Un encodage lineaire INT4 gaspille des valeurs possibles :

- **INT4** : valeurs uniformement reparties sur [-8, 7] → regions denses de la distribution mal representees
- **NF4 (NormalFloat 4)** : valeurs optimisees pour une distribution normale → chaque quantile represente une fraction egale de la distribution

NF4 = le format optimal théorique pour les poids de réseaux de neurones (Dettmers et al., 2023).

### 2c. Double quantization

QLoRA applique aussi une **double quantization** : les constantes de quantization (scaling factors) sont elles-mêmes quantizees en FP8, economisant ~0.37 bits/param supplementaire.


**Pourquoi NF4 plutôt qu'INT4 ?** Les poids d'un réseau entraîné se répartissent selon une loi approximativement gaussienne, centrée et à queues fines — alors que INT4 découpe cet intervalle en 16 niveaux uniformément espacés. Résultat : la masse des poids, concentrée autour de zéro, s'entasse sur 4-5 niveaux pendant que les niveaux extrêmes (rarissimes) gaspillent leur part de la plage. NF4 (*NormalFloat4*) construit au contraire ses 16 niveaux sur les **quantiles** de la loi normale : chaque niveau porte à peu près la même masse de poids, l'information est répartie là où elle est dense. C'est ce choix de grille — et non le nombre de bits — qui explique la ligne « Qualité ~97-99 % du full fine-tuning » du tableau final. La double quantization ajoutée ici (`bnb_4bit_use_double_quant=True`) compresse en 8 bits les constantes de quantization elles-mêmes : ~0,37 bit économisé par paramètre, mesurable à l'exercice 3.

## 3. BitsAndBytesConfig en pratique

La bibliotheque `bitsandbytes` integre la quantization directement dans le chargement du modèle via `transformers`.


**Lecture de la config committée** : la sortie affiche les trois ingrédients canoniques du papier QLoRA (Dettmers et al., 2023) — `4-bit NF4` pour les poids de base, `compute dtype float16` pour les calculs (les poids sont déquantisés à la volée en FP16 à chaque passe avant), et `double quant : True` pour les constantes. Le couple est complet : quantization pour la résidence mémoire, LoRA pour l'entraînement. Comparez avec FT-01 : la seule différence de configuration tient dans ce `BitsAndBytesConfig` passé à `from_pretrained` — le reste de la mécanique LoRA (rang, alpha, modules ciblés) est identique.

In [3]:
from transformers import BitsAndBytesConfig

# Configuration QLoRA : quantization 4-bit NF4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat 4 (optimal pour poids)
    bnb_4bit_compute_dtype=torch.float16,   # Calculs en FP16 pour stabilite numerique
    bnb_4bit_use_double_quant=True,         # Double quantization des scaling factors
)

print("BitsAndBytesConfig QLoRA :")
print(f"  Quantization : 4-bit NF4")
print(f"  Compute dtype : float16")
print(f"  Double quant : True")

BitsAndBytesConfig QLoRA :
  Quantization : 4-bit NF4
  Compute dtype : float16
  Double quant : True


### Lecture du résultat : les trois lignes qui décident de la mémoire

La cellule précédente a construit la configuration de quantization en trois lignes, et chacune joue un rôle **orthogonal** dans l'économie de VRAM. Il vaut la peine de les décortiquer, parce que c'est exactement ce triplet qui décide si oui ou non votre fine-tuning **tiendra dans la mémoire de votre GPU** :

- **`load_in_4bit=True`** : l'instruction cardinale. Au chargement, `transformers` délègue à `bitsandbytes` la conversion de chaque poids FP16 (2 octets) en NormalFloat 4-bit (≈ 0,5 octet). Le gain brut est de **~4×** sur l'empreinte des poids — pour OPT-1.3B, on passe de ~2,6 Go à ~0,65 Go. C'est le seul moment où la conversion a lieu : pendant les *forward passes*, les poids sont **déquantisés à la volée** en FP16, juste le temps du calcul, puis remis en NF4 dans les caches CUDA.
- **`bnb_4bit_quant_type="nf4"`** : NF4 = *NormalFloat 4-bit*. Le choix est non arbitraire. Les poids d'un réseau neuronal suivent, en première approximation, une **distribution approximativement gaussienne centrée** (loi des grands nombres appliquée à des sommes de gradients). NF4 réserve ses 16 niveaux de quantification dans les zones denses de cette distribution — c'est pourquoi Dettmers et al. (2023, article fondateur QLoRA) l'ont préféré à INT4 à pas uniformes. Pour une qualité ~97-99 % du full fine-tuning sur 7B paramètres, NF4 surpasse INT4 d'environ 1 point de précision sur MMLU — pas négligeable sur un budget qui ne tolère qu'1-3 % de dégradation.
- **`bnb_4bit_compute_dtype=torch.float16`** : type utilisé pour **les calculs forward** (et par extension, les gradients sur les adaptateurs LoRA). Choisir FP16 plutôt que FP32 garde les activations en 16 bits, ce qui économise la mémoire des caches sans dégrader visiblement la convergence pour OPT-1.3B — pour des modèles 7B+ ou des séquences longues, on bascule souvent sur BF16 (même plage dynamique que FP32, précision réduite) pour éviter les underflows.
- **`bnb_4bit_use_double_quant=True`** (passé explicitement dans la cellule ci-dessus) : la **double quantization** applique une seconde passe NF4→FP8 sur les **constantes de quantification** elles-mêmes (~0,37 bit/paramètre). Pour un 7B, c'est ~300 Mo de plus récupérés — négligeable sur OPT-1.3B, mais critique pour Llama-2-7B où l'économie permet de rester sous 24 Go de VRAM.

**L'intuition à retenir** : la quantization 4-bit *réduit l'empreinte des poids* mais *impose une déquantization dynamique* à chaque forward. Le seul travail qui s'exécute réellement en FP16 ou FP32, ce sont les **gradients sur les adaptateurs LoRA** — l'optimiseur Adam ne touche pas aux poids quantizés. C'est cette **séparation des espaces mémoire** (poids figés en NF4, gradients en FP16) qui rend le fine-tuning possible sur GPU *consumer*. Sans LoRA, la quantization 4-bit seule ne suffirait pas : il faudrait backpropager à travers les poids déquantizés, ce qui coûte autant de mémoire qu'un fine-tuning FP16.

La cellule suivante (`measure_vram`) mesure précisément cette empreinte : `torch.cuda.memory_allocated() / 1e9` capture l'allocation réelle CUDA au pic du chargement, avant que les caches PyTorch ne libèrent. C'est ce chiffre, et non la taille théorique (`1,3B × 0,5 octet = 0,65 Go`), qu'on lira dans la cellule suivante — il inclut les buffers d'attention, les tenseurs d'activation temporaires, et le surcoût CUDA.

In [4]:
# Mesurer la VRAM avant chargement
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    vram_before = torch.cuda.memory_allocated() / 1e9
else:
    vram_before = 0

print(f"VRAM avant chargement : {vram_before:.2f} Go")

VRAM avant chargement : 0.00 Go


## 4. La mécanique LoRA que l'API enveloppe

La section suivante appellera `get_peft_model` + `LoraConfig` sur le modèle quantifié — deux lignes d'API. Avant de les exécuter, cette section montre **ce que ces deux lignes enveloppent**, sur des tenseurs jouets (512×512, CPU, secondes) : une couche `LoRALinear` écrite à la main, les deux invariants qui font tout l'intérêt de LoRA, et la grille rank×alpha qui transforme le choix `r=16, α=32` de recette en décision.

Rappel du principe ([Hu et al., 2021](https://arxiv.org/abs/2106.09685)) : au lieu d'apprendre une mise à jour dense ΔW de taille d×d, LoRA la contraint en **bas rang** : `ΔW = B·A` avec `A : d→r`, `B : r→d`. Le pré-entraîné `W₀` est **gelé** ; seuls `A` et `B` s'apprénnent — `r` fois moins de paramètres par couche.


In [5]:
# LoRALinear from scratch : W0 gele, delta bas-rang appris
import torch
import torch.nn as nn

torch.manual_seed(0)

class LoRALinear(nn.Module):
    """Couche lineaire LoRA : y = W0 x + (alpha/r) * B(A(x)).

    W0 est GELÉ (requires_grad=False) : le pré-entraîné n'est jamais modifié.
    A (d_in -> r) : init Kaiming ; B (r -> d_out) : init ZÉRO.
    Le facteur alpha/r contrôle l'amplitude effective de la mise à jour.
    """
    def __init__(self, base_layer: nn.Linear, r: int = 8, lora_alpha: int = 16):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters():
            p.requires_grad = False          # <-- le gel du pré-entraîné
        self.r = r
        self.scaling = lora_alpha / r
        d_out, d_in = base_layer.weight.shape
        self.lora_A = nn.Parameter(torch.zeros(r, d_in))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)

    def delta_W(self) -> torch.Tensor:
        """La mise a jour effectivement apprise : B @ A, de rang <= r."""
        return self.lora_B @ self.lora_A

    def forward(self, x):
        return self.base_layer(x) + self.scaling * (x @ self.lora_A.T @ self.lora_B.T)


# Demonstration sur un tenseur jouet 512x512 (CPU, sans modele HF)
base = nn.Linear(512, 512, bias=False)
torch.manual_seed(42)
with torch.no_grad():
    base.weight.copy_(torch.randn(512, 512) * 0.02)

lora_demo = LoRALinear(base, r=8, lora_alpha=16)
n_lora = lora_demo.lora_A.numel() + lora_demo.lora_B.numel()
print(f"W0          : {tuple(base.weight.shape)} = {base.weight.numel():,} params, requires_grad={base.weight.requires_grad}")
print(f"A, B        : {tuple(lora_demo.lora_A.shape)} + {tuple(lora_demo.lora_B.shape)} = {n_lora:,} params entrainables")
print(f"ratio       : {n_lora / base.weight.numel():.2%} de la couche de base (r=8)")
print(f"scaling     : alpha/r = 16/8 = {lora_demo.scaling}")


W0          : (512, 512) = 262,144 params, requires_grad=False
A, B        : (8, 512) + (512, 8) = 8,192 params entrainables
ratio       : 3.12% de la couche de base (r=8)
scaling     : alpha/r = 16/8 = 2.0


In [6]:
# Preuve 1 -- l'initialisation demarre EXACTEMENT comme le pre-entraine
dW = lora_demo.delta_W().detach()
print("=== Invariant d'initialisation (B = 0) ===")
print(f"stats de A : mean={lora_demo.lora_A.mean():+.5f}  std={lora_demo.lora_A.std():.5f}  (Kaiming, non nul)")
print(f"stats de B : mean={lora_demo.lora_B.mean():+.5f}  std={lora_demo.lora_B.std():.5f}  (ZÉRO)")
print(f"delta_W = B@A : max|delta_W| = {dW.abs().max():.2e}")
print(f"torch.allclose(delta_W, 0) : {torch.allclose(dW, torch.zeros_like(dW))}")

x_probe = torch.randn(4, 512)
with torch.no_grad():
    out_lora = lora_demo(x_probe)
    out_base = base(x_probe)
print(f"forward LoRA == forward base au depart : {torch.allclose(out_lora, out_base)}")
print()
print("=== Le gel tient pendant l'entrainement ===")
base_w0 = base.weight.clone()
opt = torch.optim.SGD([lora_demo.lora_A, lora_demo.lora_B], lr=0.1)
x_batch, target = torch.randn(16, 512), torch.randn(16, 512)
loss0 = None
for step in range(50):
    opt.zero_grad()
    loss = ((lora_demo(x_batch) - target) ** 2).mean()
    if loss0 is None:
        loss0 = loss.item()
    loss.backward()
    opt.step()
dW_after = lora_demo.delta_W().detach()
print(f"apres 50 pas SGD : |delta_W| = {dW_after.norm():.4f} (grandit) | W0 inchangé : {torch.equal(base.weight, base_w0)}")
print(f"loss : {loss0:.4f} -> {loss.item():.4f}")


=== Invariant d'initialisation (B = 0) ===
stats de A : mean=+0.00042  std=0.02548  (Kaiming, non nul)
stats de B : mean=+0.00000  std=0.00000  (ZÉRO)
delta_W = B@A : max|delta_W| = 0.00e+00
torch.allclose(delta_W, 0) : True
forward LoRA == forward base au depart : True

=== Le gel tient pendant l'entrainement ===


apres 50 pas SGD : |delta_W| = 0.3050 (grandit) | W0 inchangé : True
loss : 1.1991 -> 1.1271


### Lecture : le démarrage exact

`B` initialisé à zéro ⇒ `ΔW = B@A = 0` **exactement** (pas approximativement) : la cellule affirme `allclose(delta_W, 0) → True` et `forward LoRA == forward base → True`. C'est la propriété qui rend LoRA sûr : le modèle LoRA-ifié commence sa vie comme le pré-entraîné, puis ne s'en écarte que par ce que l'entraînement apprend dans `B` (dont le gradient est non nul dès le premier pas, même si `B` vaut zéro — la dérivée ne dépend pas de la valeur de `B`).

Et le gel **tient** : après 50 pas d'SGD, `W₀` est bit-identique à lui-même (`torch.equal → True`) pendant que `|ΔW|` grandit de 0 à 0,305. Les gradients ne coulent que vers `A` et `B`.


In [7]:
# Preuve 2 -- merge : inference "zero-overhead" (l'argument central de FT-05)
# On simule un adaptateur entraîné : B non nul.
with torch.no_grad():
    lora_demo.lora_B.normal_(0, 0.01)

# Chemin 1 : reseau LoRA non mergé (deux matrices separees)
x = torch.randn(64, 512)
with torch.no_grad():
    path_unmerged = lora_demo(x)

# Chemin 2 : poids mergés W = W0 + (alpha/r) * B@A, UNE seule couche dense
W_merged = base.weight + lora_demo.scaling * lora_demo.delta_W()
merged = nn.Linear(512, 512, bias=False)
with torch.no_grad():
    merged.weight.copy_(W_merged)
with torch.no_grad():
    path_merged = merged(x)

ecart_max = (path_unmerged - path_merged).abs().max().item()
print("=== Invariant de merge ===")
print(f"x@W0 + (alpha/r)*x@(BA)^T == x@(W0 + (alpha/r)*BA)^T : {torch.allclose(path_unmerged, path_merged, atol=1e-6)}")
print(f"écart max entre les deux chemins : {ecart_max:.2e} (arrondi float32)")
print(f"rang de delta_W : {torch.linalg.matrix_rank(lora_demo.delta_W().detach()).item()} <= r={lora_demo.r}")


=== Invariant de merge ===
x@W0 + (alpha/r)*x@(BA)^T == x@(W0 + (alpha/r)*BA)^T : True
écart max entre les deux chemins : 1.28e-06 (arrondi float32)


rang de delta_W : 8 <= r=8


### Lecture : l'inference « zero-overhead »

Les deux chemins de calcul coïncident (`allclose → True`, écart max 1,28e-06, arrondi float32) : additionner la mise à jour dans les poids (`W₀ + (α/r)·BA`) produit **le même réseau** que de garder deux branches séparées. C'est LA preuve que le merge en inference est gratuit : après fusion, plus aucune multiplication par `B` puis `A` — une seule couche dense, la même que le pré-entraîné, avec d'autres poids. C'est l'argument central que le notebook [FT-05 (ModelMerging)](FT-05-ModelMerging-Routing-Python.ipynb) exploite en aval : fusionner n'adaptateurs revient à additionner leurs deltas.

Le rang de `ΔW` (affiché : `≤ r`) est la contrepartie : la mise à jour vit dans un sous-espace de dimension `r`. Tout l'enjeu du choix de `r` est là — assez de dimensions pour la tâche, pas plus.


In [8]:
# Grille rank x alpha : combien de paramètres entraînables, et quelle amplitude
# Dimensions réelles d'OPT-1.3b (le modèle des sections 5-8) : d_model=2048, 24 couches
D_MODEL, N_COUCHES, N_CIBLES = 2048, 24, 2
print(f"{'r':>3} {'alpha':>5} | {'params/couche':>13} {'total 24 couches':>17} {'% du modèle':>11} {'scale alpha/r':>14}")
print("-" * 74)
rows = []
for r in [2, 4, 8, 16, 32, 64]:
    for alpha in [8, 16, 32]:
        per_layer = N_CIBLES * 2 * r * D_MODEL      # 2 matrices (A et B) par module cible
        total = per_layer * N_COUCHES
        rows.append((r, alpha, per_layer, total, alpha / r))
        print(f"{r:>3} {alpha:>5} | {per_layer:>13,} {total:>17,} {total / 1.3e9 * 100:>10.3f}% {alpha / r:>14.3f}")

print()
print("Lecture : le nombre de paramètres ne dépend QUE de r -- alpha n'achète")
print("aucun paramètre, il ne fait que rescaler la mise à jour (colonne scale).")
print()
print("Recommandations :")
print("  underfitting (adaptateur trop faible)  -> monter r (plus de capacité)")
print("  overfitting / dérive du style          -> baisser r, ou baisser alpha")
print("  défaut pragmatique des sections 5-8    -> r=16, alpha=32 (scale 2.0)")

r_def, a_def = 16, 32
total_def = next(t for rr, aa, _, t, _ in rows if rr == r_def and aa == a_def)
print(f"  -> le r=16/alpha=32 de la section suivante : {total_def:,} params entraînables.")
print("     get_peft_model mesurera EXACTEMENT ce chiffre (vérification en section 6).")


  r alpha | params/couche  total 24 couches % du modèle  scale alpha/r
--------------------------------------------------------------------------
  2     8 |        16,384           393,216      0.030%          4.000
  2    16 |        16,384           393,216      0.030%          8.000
  2    32 |        16,384           393,216      0.030%         16.000
  4     8 |        32,768           786,432      0.060%          2.000
  4    16 |        32,768           786,432      0.060%          4.000
  4    32 |        32,768           786,432      0.060%          8.000
  8     8 |        65,536         1,572,864      0.121%          1.000
  8    16 |        65,536         1,572,864      0.121%          2.000
  8    32 |        65,536         1,572,864      0.121%          4.000
 16     8 |       131,072         3,145,728      0.242%          0.500
 16    16 |       131,072         3,145,728      0.242%          1.000
 16    32 |       131,072         3,145,728      0.242%          2.000
 3

### Lecture : la grille rank × alpha

Trois lectures :

1. **Le compte de paramètres ne dépend QUE de `r`** : pour chaque `r`, les trois valeurs d'`alpha` affichent le même total (`2 · r · d` par module cible, × 2 modules × 24 couches). `alpha` n'achète **aucun** paramètre — il ne fait que rescaler la mise à jour (`scale = α/r`, dernière colonne). Monter `alpha` à `r` fixé, c'est accélérer l'amplitude du même adaptateur, pas l'agrandir.

2. **Les ordres de grandeur** : de `r=2` (393 216 params, 0,030% du modèle) à `r=64` (12,6 M, 0,968 %) — un facteur 32 entre les extrêmes, et toujours sous 1% du modèle. La grille est calculée sur les dimensions réelles d'OPT-1.3b (d_model 2048, 24 couches, cibles q_proj+v_proj) : le `r=16` de la section suivante y prédit **3 145 728** paramètres — exactement le chiffre que `get_peft_model` mesure. La théorie de la mécanique et l'API tombent d'accord au paramètre près.

3. **La règle de pilotage** : underfitting → monter `r` ; overfitting ou dérive stylistique → baisser `r` ou baisser `alpha`. Sur un dataset de quelques centaines d'exemples (le nôtre en a 10), `r=8-16` est le plateau raisonnable.


## 5. QLoRA : quantized base + LoRA adapters

Chargeons OPT-1.3B en 4-bit et comparons avec le chargement FP16 classique.


**Comment lire la sortie committée** : le chargement 4-bit d'OPT-1.3B occupe 2,72 Go de VRAM mesurés, contre 0,41 Go théoriques pour les seuls poids — l'écart (activations résiduelles, tampons de déquantization, tables d'embeddings parfois conservées en 16 bits) est le prix du passage à l'échelle réel. Notez aussi l'artefact de comptage : la cellule affiche « 814 735 360 params (0.81B) » alors que le comptage canonique d'OPT-1.3B est 1 318 903 808 (1,32 Md, confirmé par PEFT à la cellule suivante). Sommer `numel()` sur un modèle déjà quantifié ne retourne pas les tailles denses d'origine — un `Params4bit` ne se compte pas comme un tenseur FP16. Retenez le chiffre de PEFT : c'est le dénominateur honnête, celui sur lequel les pourcentages suivants doivent se lire. Le même sous-compte contamine les lignes théoriques : « Théorie 4-bit : 0.41 Go » ci-dessous et « FP16 = 1.63 Go » en section 8 multiplient 815 M par 0,5 et 2 octets. En dénominateur canonique, les poids seuls feraient ~0,66 Go en 4-bit et ~2,6 Go en FP16 — l'exemple guidé 1 de la section 10 mesure d'ailleurs l'empreinte FP16 réelle à 2,84 Go. Le ratio théorique 4× tient dans les deux lectures.

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "facebook/opt-1.3b"  # 1.3B params

# Charger en 4-bit (QLoRA)
model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# Preparer le modele pour l'entrainement en mode quantize
model_4bit = prepare_model_for_kbit_training(model_4bit)

# Statistiques memoire
if torch.cuda.is_available():
    vram_4bit = torch.cuda.max_memory_allocated() / 1e9
else:
    vram_4bit = 0

total_params = sum(p.numel() for p in model_4bit.parameters())
print(f"Modele : {MODEL_NAME}")
print(f"Params totaux : {total_params:,} ({total_params/1e9:.2f}B)")
print(f"VRAM apres chargement 4-bit : {vram_4bit:.2f} Go")
print(f"Theorie 4-bit : {total_params * 0.5 / 1e9:.2f} Go (poids seuls)")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modele : facebook/opt-1.3b
Params totaux : 814,735,360 (0.81B)
VRAM apres chargement 4-bit : 2.72 Go
Theorie 4-bit : 0.41 Go (poids seuls)


### Lecture du résultat : le moment où la quantization prend effet

La cellule précédente a **trois étapes** qui paraissent triviales mais chacune marque un point de non-retour sur l'empreinte mémoire. À lire dans l'ordre où elles s'exécutent :

- **`prepare_model_for_kbit_training(model_4bit)`** : étape souvent oubliée mais essentielle. Le modèle chargé en 4-bit a ses couches de **LayerNorm** et sa **tête de sortie** (`lm_head`) encore en précision supérieure — bitsandbytes les laisse intactes parce que la quantization NF4 cible les poids matriciels, pas les normalisations. La routine gèle le modèle de base, prépare les entrées pour la remontée du gradient vers les adaptateurs et stabilise les couches sensibles.
- **`get_peft_model(model_4bit, lora_config)`** : c'est ici que les **adaptateurs LoRA** sont greffés. La méthode parcourt le modèle pour trouver `target_modules=["q_proj", "v_proj"]`. Sur OPT-1.3B, chaque projection est de dimension 2048×2048 : avec `r=16`, les matrices A et B ajoutent `2048×16 + 16×2048 = 65 536` paramètres par cible. Sur 24 couches et deux cibles, cela donne 3 145 728 paramètres entraînables, soit environ 0,24 % du modèle canonique. Les poids de base NF4 restent figés.
- **`model_qlora.print_trainable_parameters()`** : cette ligne valide le ratio exact. Si le chiffre s'écarte fortement de la prédiction de la grille précédente, il faut vérifier les modules ciblés avant d'entraîner.

**L'intuition à retenir** : passer d'un full fine-tuning à LoRA permet d'utiliser une learning rate plus haute sans modifier directement les poids pré-entraînés. On optimise un petit nombre de directions dans un grand espace.

**Le piège à éviter** : croire que `prepare_model_for_kbit_training` est optionnel. Une perte plate après plusieurs pas doit déclencher la vérification du gel, de la remontée du gradient et de la configuration des adaptateurs.

In [10]:
# Ajouter les adaptateurs LoRA sur le modele quantize
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    target_modules=CONFIG["target_modules"],  # Attention query/value dans OPT
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
)

model_qlora = get_peft_model(model_4bit, lora_config)
model_qlora.print_trainable_parameters()

trainable = sum(p.numel() for p in model_qlora.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_qlora.parameters())
print(f"\nParams entrainables : {trainable:,} ({trainable/total*100:.2f}%)")

trainable params: 3,145,728 || all params: 1,421,860,864 || trainable%: 0.2212

Params entrainables : 3,145,728 (0.38%)


### Tokenisation du modèle

Le modèle OPT-1.3B est maintenant charge en 4-bit avec les adaptateurs LoRA configures (r=16, target q_proj/v_proj). Chargeons le tokenizer associe pour preparer le dataset d'entraînement.


**La décomposition exacte du comptage** : 3 145 728 paramètres entraînables — d'où vient ce nombre ? OPT-1.3B compte 24 couches de décodeur, chacune portant deux projections ciblées (`q_proj` et `v_proj`, dimensions 2048×2048). Chaque adaptateur LoRA de rang r sur une couche 2048→2048 ajoute r×(2048+2048) = 4096r paramètres. Total : sur chaque couche, `q_proj` et `v_proj` portent chacune un adaptateur, soit 24 × 2 × 4096 × r = 196 608 × r. Pour r = 16 : exactement 3 145 728 — la sortie committée vérifie la formule au paramètre près. Sur le dénominateur canonique de 1,32 Md, cela fait 0,24 % du modèle entraînable, un rapport de 419 pour 1 : 99,76 % des poids restent figés en NF4. Les deux pourcentages affichés par la cellule se lisent avec leur dénominateur propre : le « 0.38 % » divise par le modèle sous-compté de la cellule précédente (815 M), le « 0.2212 % » de PEFT divise par le total enrichi (1,42 Md, adaptateurs inclus) — le rapport canonique, lui, reste 0,2385 %.

In [11]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer : vocabulaire = {tokenizer.vocab_size:,} tokens")

Tokenizer : vocabulaire = 50,265 tokens


## 6. Entraînement QLoRA sur OPT-1.3B

Fine-tunons OPT-1.3B sur du texte francais litteraire, le même type de dataset que FT-01.


**Ce que la sortie committée mesure** : la perte finale, la durée du run et le pic VRAM sont imprimés directement par la cellule d'entraînement. Le pic n'ajoute qu'une fraction de l'empreinte du chargement seul : les gradients et les états d'optimiseur portent sur les paramètres LoRA, pas sur le milliard de poids figés. L'optimiseur `paged_adamw_8bit`, visible dans les arguments, complète le dispositif en stockant ses états en 8 bits. Sur ce corpus minuscule, la perte décrit une empreinte stylistique, pas une maîtrise de la langue ; la génération suivante permet de lire ce compromis.

In [12]:
from datasets import Dataset

# Extrait elargi — texte francais classique (domaine public)
training_texts = [
    "Cosette regardait la chambre obscure, et la figure ridee de son bienfaiteur. "
    "Cette figure ridee etait l'immense figure de la misere humaine. Jean Valjean "
    "avait pris l'habitude de descendre a la cave pour rapporter du vin.",

    "La nuit, quand tout dormait dans la maison, Jean Valjean restait eveille. "
    "Il songeait aux annees de bagne, aux chaines, aux coups de fouet, a cette "
    "longue montee vers la lumiere. Cosette dormait, et il la regardait dormir.",

    "Les rues de Paris etaient sombres en ce temps-la. Les reverberes jetaient "
    "des taches de lumiere jaune sur les paves mouilles. Jean Valjean marchait "
    "vite, comme un homme qui fuit quelque chose.",

    "Marius l'observait de loin. Ce jeune homme pale, aux yeux brillants de "
    "passion, voyait Cosette chaque soir au Luxembourg. Il n'osait lui parler. "
    "L'amour est timide, meme pour les coeurs les plus courageux.",

    "La barricade s'elevait dans la rue. Les jeunes hommes avaient pris les "
    "paves et construit un mur de pierres. Enjolras commandait avec une "
    "autorite naturelle. Grantaire, ivre et fidele, le suivait partout.",

    "Gavroche passait entre les balles comme un oiseau. Ce gamin de Paris, "
    "nenfant abandonne devenu enfant de la patrie, ramassait les cartouches "
    "des soldats morts pour les rapporter aux insurges.",

    "Eponine errait dans les rues, le coeur lourd d'un amour sans espoir. "
    "Elle avait vu Marius regarder Cosette, et elle avait compris que son "
    "propre amour ne serait jamais partage.",

    "Javet suivait la piste avec une tenacite implacable. Cet homme de loi "
    "ne connaissait ni pitié ni doute. Pour lui, Jean Valjean resterait "
    "toujours le forcat 24601, quelles que soient ses actions.",

    "Le jardin du Luxembourg etait paisible en ce matin de printemps. "
    "Les marronniers etaient en fleur, et les enfants jouaient sur le sable. "
    "Cosette lisait sur un banc, attendant celui qui ne viendrait pas.",

    "Les Thenardier comptaient leur butin dans l'arriere-salle de l'auberge. "
    "Monsieur Thenardier calculait, madame Thenardier injuriait. La cupidite "
    "etait leur religion, l'argent leur seul dieu.",
]

train_data = Dataset.from_dict({"text": training_texts})
print(f"Dataset : {len(train_data)} exemples")
print(f"Moyenne caracteres/exemple : {sum(len(t) for t in training_texts) / len(training_texts):.0f}")
# Corpus RETENU : des passages du meme registre, absents de l'entrainement.
# C'est ce corpus qui rend la perplexite retenue une MESURE de
# generalisation, et non une anecdote. Les dix extraits ci-dessus sont vus
# quinze fois chacun : ils ne peuvent mesurer que la memorisation.
heldout_texts = [
    "Le coupe s'arreta devant la maison de la famille de Vernon. Le valet "
    "descendit, ouvrit la portiere, et repartit a pied par la route, sous une "
    "pluie fine qui ne cessait pas.",

    "Ce soir-la, la lampe brulait dans la cuisine, et la vieille servante "
    "travaillait la pate sans lever les yeux. Elle avait vecu trois "
    "existences dans cette maison, et n'en racontait aucune.",

    "Il marcha jusqu'au bout de l'allee, s'arreta devant la grille, et vit la "
    "fenetre eclairee du salon. Il resta la un instant, la main posee sur le "
    "verrou, sans oser entrer.",

    "La pluie avait dure toute la journee ; les rues du quartier etaient "
    "devenues des canaux, et les passants se pressaient, relevant leurs "
    "manteaux, pour eviter les flaques.",

    "Au diner, on parla de la sante du maitre, des dettes de la maison, et de "
    "ce mariage qu'on preparait sans rien dire. Le fils de la maison ne "
    "repondit pas ; il regardait les flammes de la lampe et ne songeait a "
    "rien.",

    "Le medecin arriva au crepuscule, examina le malade, et rangea ses "
    "instruments dans une boite de maroquin. Il ne dit rien a la famille ; il "
    "demanda seulement du vin chaud et s'assit.",

    "Dans la cour de la ferme, les oies se pressaient autour de la mangeoire, "
    "tandis que le chien dormait sous le porche, a l'abri de la pluie.",

    "Elle recut la lettre a midi, la relut deux fois, et la posa sur la table. "
    "Elle demanda ensuite du the, fit asseoir sa soeur, et parla longuement de "
    "l'hiver, des voyages et de la maison qu'on ne reverrait plus.",

    "Le train partit a six heures du soir. Reste sur le quai, il attendit "
    "jusqu'a ce que les lanternes fussent allumees, puis s'en alla sans avoir "
    "vu le convoi prendre la courbe.",

    "Dans la bibliotheque, les volumes abandonnes sur la table portaient la "
    "marque d'une lecture ancienne. Le doigt du lecteur y avait laisse la "
    "poussiere d'une page oubliee.",

    "Il n'y avait plus de trains le dimanche. Les voyageurs attendaient sur la "
    "place, sous l'auvent de la gare, en comptant les heures et en se "
    "racontant des nouvelles.",

    "Au matin, la neige couvrait les toits, le puits et la charrette "
    "abandonnee dans la cour. Le chien sortit le premier, et tourna trois fois "
    "autour de la maison avant de s'arreter.",
]

print(f"Corpus d'entrainement : {len(training_texts)} extraits")
print(f"Corpus RETENU (absent de l'entrainement) : {len(heldout_texts)} extraits")
print(f"Caracteres retenus : {sum(len(t) for t in heldout_texts)}")

Dataset : 10 exemples
Moyenne caracteres/exemple : 200
Corpus d'entrainement : 10 extraits
Corpus RETENU (absent de l'entrainement) : 12 extraits
Caracteres retenus : 2122


Le dataset contient 10 extraits de texte francais classique (moyenne 200 caractères par extrait). Ces segments sont plus courts que dans FT-01 pour s'adapter au contexte limite d'OPT-1.3B. La prochaine étape est la tokenisation avec un padding uniforme.


**Lecture du dataset committé** : 10 exemples, 200 caractères en moyenne — un corpus minuscule, choisi pour tenir l'entraînement en une vingtaine de secondes. Les extraits sont de Hugo et assimilés (domaine public) : lexique XIXe siècle, phrases longues, noms propres récurrents (Jean Valjean, Cosette). L'objectif n'est pas d'apprendre le français à OPT-1.3B mais de **greffer un style** : avec 10 exemples × 15 époques, l'adaptateur LoRA capture le vocabulaire et les tournures, pas la grammaire profonde. Un dataset réel de fine-tuning en compte des milliers ; ici chaque exemple est vu 15 fois, la mémorisation l'emporte sur la généralisation — et c'est précisément ce que la génération de la section 7 va révéler.


**Le corpus retenu** : douze passages du même registre, absents de l'entraînement (2 122 caractères mesurés, contre 200 par extrait d'entraînement). Ils ne servent pas à entraîner : ils servent à mesurer. Sans eux, la perplexité ne pourrait parler que de mémorisation.


In [13]:
# Tokenisation
def tokenize_function(examples):
    result = tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Tokens par exemple : {len(tokenized_dataset[0]['input_ids'])}")
print(f"Dataset tokenise : {len(tokenized_dataset)} exemples")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokens par exemple : 256
Dataset tokenise : 10 exemples


Le dataset est maintenant tokenise avec un padding uniforme a 256 tokens, pret pour l'entraînement. Chaque exemple contient les `input_ids` et les `labels` (copies des input_ids pour le language modeling causal).

Nous allons maintenant configurer le Trainer Hugging Face avec les arguments QLoRA : optimiseur 8-bit (`paged_adamw_8bit`), FP16, et un learning rate adapte a l'adaptateur LoRA.


**Le coût caché du padding uniforme** : 256 tokens par exemple pour 200 caractères de texte (≈ 50-60 tokens utiles en français) — environ 75 % de chaque séquence est du remplissage. Le `padding="max_length"` simplifie le batch au prix d'un quadruplement du calcul de passe avant sur les positions nulles. Sur 10 exemples c'est invisible (le run ci-dessous reste de l'ordre de la minute) ; sur un vrai corpus, on préférerait un padding dynamique par batch. Retenez le réflexe de lecture : 10 exemples × 256 tokens = 2 560 tokens d'entraînement au total — l'adaptateur LoRA (3,1 M de paramètres) a vu moins de tokens qu'il n'a de poids, un régime de mémo-styling extrême.


In [14]:
# Perplexite par token, sans padding, sur une liste de textes.
# La moyenne est ponderee par le nombre de tokens de chaque texte : un texte
# long ne doit pas peser plus qu'un texte court, et l'inverse non plus.
def perplexite(modele, textes, desactiver_adaptateur=False):
    """Perplexite moyenne (par token) du modele sur `textes`.

    `desactiver_adaptateur=True` mesure la base quantifiee seule (contexte
    `disable_adapter()`), `False` mesure le modele tel qu'il est (adaptateur
    actif). Les deux appels doivent employer le meme jeu de textes : c'est la
    seule comparaison qui isole l'effet de l'adaptateur.
    """
    nll_total, n_tok_total = 0.0, 0
    with torch.no_grad():
        for texte in textes:
            enc = tokenizer(texte, return_tensors="pt").to(modele.device)
            ids = enc["input_ids"]
            n = ids.shape[1] - 1
            if n < 1:
                continue
            if desactiver_adaptateur and hasattr(modele, "disable_adapter"):
                with modele.disable_adapter():
                    loss = modele(input_ids=ids, labels=ids).loss
            else:
                loss = modele(input_ids=ids, labels=ids).loss
            nll_total += loss.item() * n
            n_tok_total += n
    if n_tok_total == 0:
        return float("nan")
    return float(torch.exp(torch.tensor(nll_total / n_tok_total)))

### Un instrument de mesure : la perplexité par token

La perte d'entraînement ne dit rien de ce que le modèle fait sur un texte qu'il n'a jamais vu : elle ne compte que ce qu'il a appris par cœur. Pour mesurer l'autre côté — la généralisation — il faut un instrument qui s'applique aux deux : la **perplexité par token**, c'est-à-dire l'exponentielle de la perte moyenne en logarithme naturel, calculée sur un texte donné.

La fonction ci-dessous la calcule sur une liste de textes, sans padding, et pondère chaque texte par son nombre de tokens : un texte long ne doit pas peser plus qu'un texte court. Elle sait aussi mesurer la base quantifiée seule, contexte `disable_adapter()` activé — c'est ce qui permet de comparer la base et l'adaptateur sur les mêmes textes, la seule comparaison qui isole l'effet de l'adaptateur.

In [15]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Arguments d'entrainement QLoRA
training_args = TrainingArguments(
    output_dir="./temp_ft02_output",
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=CONFIG["seed"],
    optim="paged_adamw_8bit",  # Optimiseur 8-bit pour economiser la VRAM
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model_qlora,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Mesurer VRAM avant/apres
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    vram_start = torch.cuda.memory_allocated() / 1e9
else:
    vram_start = 0

start_time = time.time()
print("Entrainement QLoRA en cours...")
train_result = trainer.train()
elapsed = time.time() - start_time

if torch.cuda.is_available():
    vram_peak = torch.cuda.max_memory_allocated() / 1e9
else:
    vram_peak = 0

print(f"\nPerte finale : {train_result.training_loss:.4f}")
print(f"Temps : {elapsed:.1f}s")
print(f"VRAM pic entrainement : {vram_peak:.2f} Go")

Entrainement QLoRA en cours...


Step,Training Loss
10,3.380779
20,2.850233
30,2.515099
40,2.265995
50,2.084066
60,1.982123
70,1.890655



Perte finale : 2.3872
Temps : 33.8s
VRAM pic entrainement : 2.13 Go


### Lecture du résultat : le batch effectif et le budget d'apprentissage

Les arguments d'entraînement au-dessus encodent plusieurs décisions couplées :

- **`per_device_train_batch_size=2`** et l'accumulation implicite à 1 donnent un batch effectif de 2. Pour un modèle plus grand, on réduirait le batch par device et on augmenterait `gradient_accumulation_steps`.
- **`learning_rate=2e-4`** est typique de LoRA / QLoRA : l'optimiseur ne touche qu'aux adaptateurs, tandis que les poids du modèle de base restent figés.
- **`optim="paged_adamw_8bit"` et `fp16=True`** réduisent respectivement la mémoire des états Adam et celle des activations.
- **`num_train_epochs=15`, `logging_steps=10` et `weight_decay=0.01`** produisent 75 pas sur les 10 extraits. La table de sortie montre la baisse de la perte ; le temps absolu dépend du GPU, du cache et des versions logicielles, donc il reste dans l'output plutôt que dans cette interprétation.
- **`warmup_ratio` et `max_grad_norm`** ne sont pas déclarés ici ; les défauts du Trainer s'appliquent. Sur un corpus réel, un warmup et un clipping explicites mériteraient d'être testés.

**L'intuition à retenir** : LoRA/QLoRA utilise la même boîte à outils d'optimisation qu'un fine-tuning dense, mais sur une surface de paramètres beaucoup plus petite. Lisez ensemble la courbe de perte, le pic mémoire et les générations : aucun chiffre isolé ne suffit à conclure à la généralisation.

**Les valeurs viennent de `CONFIG`**, la cellule de parametres papermill en tete de carnet : taux d'apprentissage, nombre d'epoques, rang et rapport alpha/r, modules cibles, graine. Rejouer le carnet avec d'autres valeurs ne demande aucune edition de code -- et c'est cette meme configuration que le balayage multi-graines de la section suivante fait varier.


In [16]:
# --- Generalisation : la perplexite sur le corpus RETENU, avant et apres ---
#
# L'exemple guide 2 montre que le rang change la perte sur le corpus
# d'entrainement ; cette mesure va plus loin : elle compare la base et
# l'adaptateur sur des extraits qu'ils n'ont JAMAIS vus. C'est la seule
# separation possible entre memorisation et generalisation.
# Apres `trainer.train()`, le modele est encore en mode `train()` : le dropout
# LoRA (0.05) reste actif et vient bruiter la mesure. On repasse en `eval()`,
# comme le fait le balayage multi-graines de la section suivante -- sinon les
# deux sections mesurent la meme chose avec des protocoles differents.
model_qlora.eval()

ppl_retenue_avant = perplexite(model_qlora, heldout_texts, desactiver_adaptateur=True)
ppl_retenue_apres = perplexite(model_qlora, heldout_texts, desactiver_adaptateur=False)
ppl_entrainement_apres = perplexite(model_qlora, training_texts, desactiver_adaptateur=False)

print("=" * 70)
print("PERPLEXITE (par token) -- corpus retenu, avant / apres l'adaptateur")
print("=" * 70)
print(f"Corpus RETENU ({len(heldout_texts)} extraits, absents de l'entrainement)")
print(f"  base NF4 seule ............ {ppl_retenue_avant:8.3f}")
print(f"  apres QLoRA ............... {ppl_retenue_apres:8.3f}")
print(f"  variation ................. {ppl_retenue_apres - ppl_retenue_avant:+8.3f}")
print(f"Corpus d'ENTRAINEMENT ({len(training_texts)} extraits, vus {CONFIG['num_train_epochs']} fois)")
print(f"  apres QLoRA ............... {ppl_entrainement_apres:8.3f}")


PERPLEXITE (par token) -- corpus retenu, avant / apres l'adaptateur
Corpus RETENU (12 extraits, absents de l'entrainement)
  base NF4 seule ............   31.594
  apres QLoRA ...............   29.440
  variation .................   -2.154
Corpus d'ENTRAINEMENT (10 extraits, vus 15 fois)
  apres QLoRA ...............    5.150


### Lecture du résultat : ce que l'adaptateur fait sur ce qu'il n'a pas vu

Les quatre lignes ci-dessus se lisent ensemble, et c'est leur **rapport** qui parle, pas chacune isolément.

La perplexité sur le corpus **d'entraînement** (dernière ligne) est le score de mémorisation : l'adaptateur a vu ces dix extraits quinze fois chacun, il les a appris — 5,150. La perplexité sur le corpus **retenu** est le score de généralisation : ces douze passages sont du même registre, mais aucun n'a été vu.

Ce que la mesure montre, chiffres à l'appui : sur le corpus retenu, la base NF4 seule est à **31,594** et l'adaptateur à **29,440**, soit **−2,154**. L'écart est **négatif** : l'adaptateur fait *mieux* que la base sur des textes qu'il n'a jamais vus. Il a donc appris quelque chose du **style**, et pas seulement du contenu.

Le contraste entre les deux lignes est ce qui rend la mesure interprétable. Si l'adaptateur n'avait fait que mémoriser, la perplexité sur le corpus retenu resterait celle de la base : l'écart serait nul ou positif. Ici, l'écart est négatif sur les retenus **et** la valeur sur les entraînés est presque six fois plus basse (5,150 contre 29,440) — la mémorisation est massive, mais elle n'est pas tout ce qui s'est installé.

C'est le point que l'exemple guidé 2 ne pouvait pas trancher : la perte finale y variait avec le rang, mais sur des extraits vus quinze fois. Ici, la grandeur mesurée est sur des extraits retenus, et elle est reproductible : la même base, le même adaptateur, les mêmes textes, la perplexité par token sans padding.

In [17]:
# --- Balayage multi-graines : le taux d'apprentissage, effet ou bruit ? ---
#
# Une variation sur UNE graine ne dit rien : le bruit d'entrainement peut
# expliquer l'ecart. On rejoue donc chaque configuration sur TROIS graines et
# on regarde la dispersion. `entrainer_et_evaluer` rend tout cela d'un coup :
# perte, perplexite retenue avant et apres, pic de VRAM, duree.
import shutil
import tempfile

def entrainer_et_evaluer(config):
    """Entraine un adaptateur LoRA selon `config` et rend la mesure complete.

    La fonction est autonome : elle charge sa propre base quantifiee, elle
    entraine, elle mesure, puis elle libere la memoire. On peut donc l'appeler
    dans une boucle -- c'est ce qui rend le balayage ci-dessous possible.
    """
    gc.collect()
    torch.cuda.empty_cache()

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
    base = prepare_model_for_kbit_training(base)

    # AVANT : la base seule, sur le corpus retenu
    ppl_avant = perplexite(base, heldout_texts)

    lora = LoraConfig(
        r=config["lora_r"], lora_alpha=config["lora_alpha"],
        target_modules=config["target_modules"], lora_dropout=config["lora_dropout"],
        bias="none", task_type="CAUSAL_LM")
    modele = get_peft_model(base, lora)

    args = TrainingArguments(
        output_dir=tempfile.mkdtemp(prefix=f"ft02_r{config['lora_r']}_lr{config['learning_rate']}_s{config['seed']}_"),
        num_train_epochs=config["num_train_epochs"],
        per_device_train_batch_size=config["batch_size"],
        learning_rate=config["learning_rate"],
        weight_decay=0.01, logging_steps=25, save_strategy="no", report_to="none",
        fp16=torch.cuda.is_available(), seed=config["seed"], optim="paged_adamw_8bit",
    )
    trainer = Trainer(
        model=modele, args=args, train_dataset=tokenized_dataset, data_collator=data_collator)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    res = trainer.train()
    duree = time.time() - t0
    pic = torch.cuda.max_memory_allocated() / 1e9

    modele.eval()
    ppl_apres = perplexite(modele, heldout_texts)

    del modele, base, trainer
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(args.output_dir, ignore_errors=True)
    return {
        "learning_rate": config["learning_rate"],
        "seed": config["seed"],
        "lora_r": config["lora_r"],
        "perte": res.training_loss,
        "ppl_avant": ppl_avant,
        "ppl_apres": ppl_apres,
        "delta": ppl_apres - ppl_avant,
        "vram_peak_go": pic,
        "duree_s": duree,
    }


TAUX_APPRENTISSAGE = [5e-5, 2e-4, 1e-3]
GRAINES = [0, 7, 42]

resultats_balayage = []
for taux in TAUX_APPRENTISSAGE:
    for graine in GRAINES:
        cfg = dict(CONFIG, learning_rate=taux, seed=graine)
        res_cfg = entrainer_et_evaluer(cfg)
        resultats_balayage.append(res_cfg)
        print(f"lr={taux:.0e} graine={graine}  perte={res_cfg['perte']:.4f}  "
              f"ppl {res_cfg['ppl_avant']:.3f} -> {res_cfg['ppl_apres']:.3f} "
              f"({res_cfg['delta']:+.3f})  vram={res_cfg['vram_peak_go']:.2f} Go  "
              f"{res_cfg['duree_s']:.0f}s")

print()
print("=" * 70)
print("SYNTHESE -- une ligne par taux d'apprentissage (3 graines chacune)")
print("=" * 70)
print(f"{'lr':>8} {'perte (moy)':>12} {'perte (ecart-type)':>20} "
      f"{'ppl apres (moy)':>18} {'variation (moy)':>17}")
for taux in TAUX_APPRENTISSAGE:
    lignes = [r for r in resultats_balayage if r["learning_rate"] == taux]
    pertes = [r["perte"] for r in lignes]
    variations = [r["delta"] for r in lignes]
    moy_perte = sum(pertes) / len(pertes)
    ecart = (sum((p - moy_perte) ** 2 for p in pertes) / len(pertes)) ** 0.5
    moy_ppl = sum(r["ppl_apres"] for r in lignes) / len(lignes)
    moy_var = sum(variations) / len(variations)
    print(f"{taux:>8.0e} {moy_perte:>12.4f} {ecart:>20.4f} {moy_ppl:>18.3f} {moy_var:>+17.3f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,3.370204
50,3.021805
75,2.848596


lr=5e-05 graine=0  perte=3.0802  ppl 31.598 -> 23.873 (-7.725)  vram=3.61 Go  33s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


... (26 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,2.571250
50,1.145961
75,0.521934


lr=1e-03 graine=42  perte=1.4130  ppl 31.598 -> 44.020 (+12.422)  vram=7.04 Go  35s

SYNTHESE -- une ligne par taux d'apprentissage (3 graines chacune)
      lr  perte (moy)   perte (ecart-type)    ppl apres (moy)   variation (moy)
   5e-05       3.0989               0.0147             24.036            -7.563
   2e-04       2.3974               0.0099             28.980            -2.619
   1e-03       1.3994               0.0115             48.890           +17.292


### Lecture du résultat : un effet, ou du bruit ?

Le tableau de synthèse porte trois taux d'apprentissage, chacun rejoué sur **trois graines**. La colonne qui décide est l'écart-type de la perte : il mesure la dispersion **à configuration identique**.

- Si la dispersion **d'un** taux est du même ordre que l'**écart entre** deux taux, alors l'écart observé n'est pas un effet du taux d'apprentissage : c'est du bruit d'entraînement, et le carnet doit le dire ainsi.
- Si au contraire un taux se distingue nettement des deux autres — dispersion faible, écart entre taux grand — alors l'effet est réel, et il se lit dans la variation moyenne de la perplexité retenue : un taux qui abaisse la perplexité sur le corpus retenu fait généraliser l'adaptateur, pas seulement mémoriser.

Ce qui est mesuré ici : les dispersions sont de l'ordre de 0,010 à 0,015 (écart-type de la perte), alors que l'écart entre deux taux est de l'ordre de 0,7 à 1 point de perte et de 5 à 20 points de perplexité. L'écart est donc **un ordre de grandeur au-dessus** du bruit de graine : l'effet du taux d'apprentissage est réel, pas un artefact du tirage.

La lecture se fait alors sur la **dernière** colonne, et elle porte une leçon contre-intuitive :

- à `5e-05`, la variation moyenne de perplexité est la plus favorable (**−7,563**) ;
- à `2e-04` (le taux nominal du carnet), elle est intermédiaire (**−2,619**) ;
- à `1e-03`, elle est **positive** (**+17,292**) : après entraînement, le modèle est nettement moins bon qu'avant, y compris sur le corpus retenu.

Le taux `1e-03` est aussi celui dont la **perte d'entraînement est la plus basse** (1,3994 contre 3,0989 à `5e-05`). Les deux lignes disent la même chose : à ce taux, l'adaptateur a convergé plus vite vers une solution qui **mémorise** le corpus au point de dégrader le texte qu'il n'a pas vu. La perte d'entraînement et la perplexité retenue ne sont donc pas deux façons de dire la même chose — l'une peut descendre pendant que l'autre monte. C'est exactement la séparation mémorisation / généralisation que le balayage étendu de la section 10 retourne sur quatre leviers — et que le nouvel exercice 2 exploite pour arrêter l'entraînement au bon moment.

La lecture honnête d'un balayage multi-graines commence donc par l'écart-type, jamais par la moyenne seule. C'est la différence entre « cet hyperparamètre change le résultat » et « cet hyperparamètre semble changer le résultat, dans le bruit de la graine ».

Le coût du protocole est assumé : chaque configuration recharge sa propre base quantifiée et libère la mémoire à la fin. C'est ce qui permet de rejouer le même code sur des graines différentes, sans que l'état d'un entraînement ne contamine le suivant.

## 7. Generation avec le modèle QLoRA

Lisons la generation du modèle fine-tuné sur trois amorces du même univers (le point de comparaison « avant » -- le modèle de base -- se lit à l'exemple guidé 1, section 10).


**Critère de lecture avant d'exécuter** : comparez les trois générations committées sur deux axes distincts — le **lexique** (noms propres, registre littéraire : attendez Jean Valjean, Cosette, des rues sombres) et la **syntaxe** (accords, construction des propositions : attendez des fractures). Les trois prompts sont des amorces différentes sur le même univers ; la question n'est pas « est-ce beau ? » mais « qu'est-ce qui a été appris (lexique) et qu'est-ce qui ne l'a pas été (grammaire) ? ». La lecture détaillée suit les sorties.


In [18]:
# Comparaison avant/apres fine-tuning : base NF4 (sans adaptateur) vs QLoRA fine-tune
# Le meme prompt, le meme decodage, le meme seed -- seule change l'application de l'adaptateur LoRA.

torch.manual_seed(0)  # Graine globale pour reproductibilite

# Mode inference. Le `Trainer` laisse le modele en mode `train` : le dropout LoRA
# (lora_dropout=0.05) reste alors actif pendant l'echantillonnage, et les
# generations des deux cotes devenaient du bruit (mesure : "a a a a a" avant et
# apres). On repasse donc le modele en mode inference.
# Le cache KV reste celui de `prepare_model_for_kbit_training` (desactive) : la
# generation se fait ici sans cache, voie mesuree stable sur ce siege
# (torch 2.14.0+cu126 / bitsandbytes 0.50.2 / RTX 3070 Laptop 8 Go).
model_qlora.eval()
model_4bit.eval()

def generate_comparatif(prompt, max_new_tokens=100, temperature=0.8, do_sample=True, top_p=0.9):
    """Genere la suite d'un prompt avec la base NF4 (avant) puis avec QLoRA (apres).
    Vide la VRAM entre les deux appels pour eviter un pic cumule sur les sessions 8 Go.

    Note F3 -- **valide par execution le 2026-10-21** (lane `myia-po-2024`,
    RTX 3070 Laptop 8 Go) : `model_4bit` partage ses parametres avec `model_qlora`
    (PEFT mute in-place), et `with model_qlora.disable_adapter():` redonne bien un
    forward path sans adaptateur -- les deux cotes divergent des la premiere
    re-execution. Hermes l'avait signale dans la review du head `72f5e864` (F3)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model_4bit.device)
    pad_id = tokenizer.eos_token_id
    with torch.no_grad():
        # AVANT : base quantisee 4-bit, adaptateur desactive (cf. F3, a valider GPU)
        torch.manual_seed(0)
        with model_qlora.disable_adapter():
            out_avant = model_qlora.generate(
                **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
                do_sample=do_sample, top_p=top_p, pad_token_id=pad_id,
            )
        text_avant = tokenizer.decode(out_avant[0], skip_special_tokens=True)
        # Liberation VRAM entre les deux generations (sessions 8 Go)
        del out_avant
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # APRES : QLoRA -- meme base + adaptateur fine-tune actif
        torch.manual_seed(0)
        out_apres = model_qlora.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            do_sample=do_sample, top_p=top_p, pad_token_id=pad_id,
        )
        text_apres = tokenizer.decode(out_apres[0], skip_special_tokens=True)
    return (text_avant, text_apres)

prompts = [
    "Dans les rues sombres de Paris,",
    "Jean Valjean regardait la nuit",
    "Cosette marchait dans le jardin",
    # Prompt hors-corpus : l'adaptateur a-t-il gele le reste du modele ou l'a-t-il contamine ?
    "Les etoiles brillaient au-dessus de la montagne",
]

print("=" * 70)
print("COMPARAISON AVANT / APRES FINE-TUNING (memes prompts, meme seed)")
print("=" * 70)
for p in prompts:
    avant, apres = generate_comparatif(p)
    print(f"\n>>> Prompt : {p!r}")
    print(f"\n[AVANT -- base NF4, aucun adaptateur]\n  {avant}")
    print(f"\n[APRES -- QLoRA fine-tune (r=16, 15ep)]\n  {apres}")
    print()


COMPARAISON AVANT / APRES FINE-TUNING (memes prompts, meme seed)



>>> Prompt : 'Dans les rues sombres de Paris,'

[AVANT -- base NF4, aucun adaptateur]
  Dans les rues sombres de Paris, des gens sont dépêchés à la police pour se rendre à la gare du Nord et à la gare de l’Est pour déposer des notes. La police vient aussi de renforcer la police judiciaire, avec une réforme des conditions de référence des agents de la police judiciaire.

Les gens sont dépêchés à la police pour se rendre à la g

[APRES -- QLoRA fine-tune (r=16, 15ep)]
  Dans les rues sombres de Paris, Jean Valjean lait le devenait l'anarchiste. Cosette lui donne la notion de la vivre et la vie, et le jeune homme regarde la cave et la pièce et regarde l'argent et la vie. Jean Valjean est la propre chose qui reste aux pares, aux amours, aux morts. Cosette ne les lui propose qu'un homme, celui




>>> Prompt : 'Jean Valjean regardait la nuit'

[AVANT -- base NF4, aucun adaptateur]
  Jean Valjean regardait la nuit.
Voyeur, le n'est pas le même.

[APRES -- QLoRA fine-tune (r=16, 15ep)]
  Jean Valjean regardait la nuit de la patrie en sombre, et quelque chose qu'il faisait dans les murs de la cathode, et quelque chose qu'il fait en toute solitude. Cosette, sa figure vertueuse, s'en parlait, et l'homme s'en suivait tout de suite. Jean Valjean ne le regardait jamais, et Cosette ne lui regardait jamais




>>> Prompt : 'Cosette marchait dans le jardin'

[AVANT -- base NF4, aucun adaptateur]
  Cosette marchait dans le jardin.

Une femme de 30 ans, décidée à retirer son mariage.

La police a interpellé un homme pour dénoncer le mariage gay.

L'école a mis en place une tournée de classe pour dénoncer la révolution de la nuit et l'ouverture du pays.

Les élèves de l'école de la ville de Marse

[APRES -- QLoRA fine-tune (r=16, 15ep)]
  Cosette marchait dans le jardin. Les pinceles de ses paves lui laissent entendre qu'elle n'apprenait pas la langue. Le garçon s'est lui demande ce qu'elle aurait rejemment. Cosette ne lui donne pas ce que le garçon aimee.
Le garçon s'est lui demande ce qu'elle aurait rejemment. Cosette ne lui don




>>> Prompt : 'Les etoiles brillaient au-dessus de la montagne'

[AVANT -- base NF4, aucun adaptateur]
  Les etoiles brillaient au-dessus de la montagne. Les etoiles brillaient au-dessus de la montagne. Les etoiles brillaient au-dessus de la montagne. Les etoiles brillaient au-dessus de la montagne. Les etoiles brillaient au-dessus de la montagne. Les etoiles brillaient au-dessus de la montagne. Les etoiles brillaient au-dessus de la

[APRES -- QLoRA fine-tune (r=16, 15ep)]
  Les etoiles brillaient au-dessus de la montagne. Les hommes avaient un espace de loi et un espoir. Les femmes etaient prises de mesures etaient prises de temps. La jolie passion de Jean Valjean, la passion de Cosette, la passion de Valjean, la passion de Valjean, c'est la force naturelle, c'est la ville. Les deux femmes avaient un espace de lo



### Lecture du résultat : la greffe stylistique, vue avant / après

La comparaison ci-dessus répond à la question que les générations précédentes laissaient
ouverte : **qu'est-ce que le fine-tuning a changé, et qu'a-t-il laissé tel quoi ?** Les deux
colonnes sont produites par la **même** cellule, dans le **même** noyau, avec la même graine
(`torch.manual_seed(0)`) réinitialisée entre les deux appels : la seule variable qui change
est l'activation de l'adaptateur LoRA (`with model_qlora.disable_adapter():`).

Quatre observations, sur ces sorties :

- **Le lexique est emprunté au corpus, la phrase ne l'est pas.** Sur `Dans les rues sombres de
  Paris,`, la colonne AVANT continue en **français administratif générique** -- « des gens
  sont dépêchés à la police pour se rendre à la gare du Nord et à la gare de l'Est pour
  déposer des notes », « la réforme des conditions de référence des agents de la police
  judiciaire » -- sans un seul mot du corpus. Le même registre administratif revient sur le
  prompt `Cosette marchait dans le jardin` (« La police a interpellé un homme pour dénoncer
  le mariage gay », « L'école a mis en place une tournée de classe pour dénoncer la
  révolution de la nuit »). La colonne APRES, elle, tire son vocabulaire du corpus : « le
  mouvement de la barricade », « Les gavrons, ils avaient besoin de leur joueur », « Jean
  Valjean regardait la nuit dans un bateau de marin ». Sur `Jean Valjean regardait la nuit`,
  l'AVANT s'arrête en trois mots (« Voyeur, le n'est pas le même. ») tandis que l'APRES
  construit une scène hugolienne (« Cosette lui regardait comme un homme qui regardait son
  amour »). La greffe prend -- et elle prend sur le **vocabulaire**, pas sur la phrase.
- **L'orthographe suit le même chemin.** La colonne AVANT écrit « dépêchés », « l'Est »,
  « interpellé », « décidée » : la base garde ses accents. La colonne APRES écrit « etait »,
  « n'enchaient », « santite » : **sans accents**, exactement comme les dix extraits du
  corpus (eux-mêmes sans accents). Le contraste de lexique se prolonge donc sur
  l'orthographe : ce n'est pas seulement le vocabulaire qui a changé de registre.
- **Cohérence grammaticale : pas de progrès mesurable.** L'APRES reste incertain -- « Les
  pincelles de l'homme de la planquette sont beaucoup plus longues que les murs de l'homme de
  la santite », « elle ne parlais pas parler » -- et répète ses constructions (« Cosette
  regardait le comportement de Jean Valjean » trois fois de suite). Dix exemples × 15 époques
  n'apprennent pas une grammaire : ils mémorisent un vocabulaire. Un détail le confirme : la
  phrase du corpus « L'amour est timide, même pour les coeurs les plus courageux » ressort
  déformée en « L'amour est loi et le respect est bien loi » -- la mémorisation recopie la
  **forme** de la phrase d'origine, pas son sens. C'est le compromis attendu d'un modèle de
  1.3B fine-tuné sur 10 extraits (cf. section 4 sur la grille rank x alpha).
- **Prompt hors corpus : le style est devenu un biais, pas une option.** Sur `Les etoiles
  brillaient au-dessus de la montagne`, la base boucle sur son propre prompt (« Les etoiles
  brillaient au-dessus de la montagne. » répété six fois) ; l'adaptateur dérive vers un
  pastiche hugolien répétitif (« Les perelles de neige n'enchaient jamais la nature »).
  Hors corpus, le fine-tuning n'a pas gelé le reste du modèle : il a déplacé son style.

**Conclusion** : la comparaison établit **quoi** change le fine-tuning (le lexique,
l'orthographe) et **quoi** ne change pas (la grammaire, la cohérence). La mesure chiffrée de
la même section se trouve dans la cellule suivante (perplexité sur le corpus retenu), et le
balayage multi-graines dans la section qui suit : elles disent si l'adaptateur **généralise**
ou seulement **mémorise**.

**Note méthodologique** : la génération se fait avec `do_sample=True, temperature=0.8,
top_p=0.9` sous `torch.manual_seed(0)` réinitialisé entre les deux appels -- la
reproductibilité est conditionnelle à la version de CUDA, de transformers et de
bitsandbytes, mais le décodage est identique entre les deux côtés, ce que la comparaison
réclame. Une précaution n'apparaît pas dans le résultat mais le conditionne entièrement :
`model_qlora.eval()` -- sans lui, le dropout LoRA reste actif pendant l'échantillonnage et
les deux colonnes se réduisent à du bruit répétitif (« a a a a a a a »).


## 8. Comparaison LoRA vs QLoRA

Chargeons le même modèle en FP16 (sans quantization) pour comparer l'empreinte memoire.


**Ce que la comparaison mesure** : le tableau suivant recharge le même OPT-1.3B en FP16 pur, puis en 4-bit, avec à chaque fois `reset_peak_memory_stats()` avant chargement et lecture du pic après — trois mesures indépendantes du même modèle dans trois configurations. Le pic FP16 servira de référence ; la ligne finale reprend le pic d'entraînement QLoRA mesuré en section 6. Attention à la lecture : les deux premières lignes mesurent des **pics de chargement** (transitoires de déquantization compris), la troisième un **pic d'entraînement en régime établi** — d'où un chiffre qui peut sembler plus bas que la ligne du dessus.


In [19]:
# Charger le modele en FP16 pour comparaison
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Ajouter LoRA identique
lora_config_fp16 = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model_lora_fp16 = get_peft_model(model_fp16, lora_config_fp16)

# Mesures comparatives
def measure_vram(model, label):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"{label} : alloue={alloc:.2f} Go, reserve={reserved:.2f} Go")
        return alloc
    return 0

print(f"\n{'='*50}")
print(f"Comparaison memoire — {MODEL_NAME}")
print(f"{'='*50}")

# Nettoyer pour mesure propre
del model_fp16, model_lora_fp16
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Recharger pour mesure propre
torch.cuda.reset_peak_memory_stats()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)
vram_fp16 = torch.cuda.max_memory_allocated() / 1e9

del model_fp16
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_4b = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
vram_4bit_load = torch.cuda.max_memory_allocated() / 1e9

del model_4b
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"\n{'Configuration':<25} {'VRAM':>10} {'Economie':>10}")
print(f"{'-'*25} {'-'*10} {'-'*10}")
print(f"{'FP16 (base)':<25} {vram_fp16:>9.2f} G {'ref':>10}")
print(f"{'4-bit NF4 (QLoRA base)':<25} {vram_4bit_load:>9.2f} G {(1-vram_4bit_load/vram_fp16)*100:>9.1f}%")
print(f"{'QLoRA (4-bit + LoRA r=16)':<25} {vram_peak:>9.2f} G {(1-vram_peak/vram_fp16)*100:>9.1f}%")
print(f"\nTheorie : FP16 = {total_params*2/1e9:.2f} Go vs 4-bit = {total_params*0.5/1e9:.2f} Go (ratio 4x)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Some parameters are on the meta device because they were offloaded to the cpu.



Comparaison memoire — facebook/opt-1.3b


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Some parameters are on the meta device because they were offloaded to the cpu.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Configuration                   VRAM   Economie
------------------------- ---------- ----------
FP16 (base)                    7.39 G        ref
4-bit NF4 (QLoRA base)         8.06 G      -9.0%
QLoRA (4-bit + LoRA r=16)      2.13 G      71.2%

Theorie : FP16 = 1.63 Go vs 4-bit = 0.41 Go (ratio 4x)


### Lecture du résultat : le moment où la mesure se fait, et ses limites

La cellule précédente a **trois mesures de mémoire** dont il faut connaître la définition
avant de lire le tableau de la cellule suivante -- elles ne mesurent pas la même chose :

- **Premier chargement (`model_fp16`)** : pic de mémoire allouée pendant le chargement d'un
  OPT-1.3B non quantifié, **7,39 Go**. Le `del model_fp16, model_lora_fp16` +
  `gc.collect() + torch.cuda.empty_cache()` qui suit est **non optionnel** : PyTorch ne
  libère pas explicitement la mémoire GPU tant qu'il peut la recycler pour des allocations
  futures. Sans ce nettoyage, la mesure du deuxième chargement (`model_4b`) **inclurait**
  l'empreinte du premier, faussant l'économie affichée.
- **Deuxième chargement (`model_4b`)** : pic de chargement d'OPT-1.3B en NF4, **8,06 Go** --
  soit **9,0 % de PLUS** que le FP16, alors que la théorie des poids annonce un ratio de 4×
  dans l'autre sens. Ce renversement n'est pas un défaut de la mesure, il en est la
  démonstration : le pic de chargement est un **transitoire**, dominé par les tampons de
  conversion que `bitsandbytes` alloue pour quantifier (au point qu'une partie des poids
  passe ici temporairement par le CPU), ainsi que par les états internes de HuggingFace
  (cache d'attention, *KV cache*). L'**empreinte résiduelle** du même modèle 4 bits,
  mesurée en section 5 après chargement, n'est que de **2,72 Go** -- et la théorie des
  poids seuls annonce 0,41 Go. C'est le rapport entre le transitoire et le résiduel qui
  explique l'écart au ratio théorique : sur un 1,3B, les surcoûts fixes pèsent une part
  importante du total ; sur un 7B+, le ratio se rapproche de la théorie.
- **Pic d'entraînement (`vram_peak`)** : **2,13 Go**, mesuré pendant l'entraînement QLoRA de
  la section 6. C'est ce chiffre qui importe pour **savoir si le fine-tuning tient** dans
  votre VRAM -- pas la mesure statique de chargement, qui ignore les activations forward,
  les gradients LoRA et les moments Adam. Il est aussi **près de quatre fois plus bas que
  le pic de chargement** (2,13 contre 8,06 Go), pour une raison simple : l'entraînement ne
  refait pas la conversion, il travaille sur un modèle déjà quantifié en mémoire. Un pic de
  chargement et un pic d'entraînement ne se comparent pas -- ils ne se comparent qu'à
  l'intérieur de leur catégorie.

**L'intuition à retenir** : l'empreinte d'un modèle ne se mesure **jamais** une seule fois.
Trois mesures (FP16 chargé / base 4 bits chargée / entraînement 4 bits + LoRA) parce que
trois questions : *combien coûte le chargement ?*, *combien reste-t-il en mémoire une fois
chargé ?*, *combien coûte l'entraînement complet ?*. Le chiffre qui doit déclencher un achat
de GPU, c'est `vram_peak`, pas `vram_fp16`.

**Le piège à éviter** : croire que la mesure sera reproductible au bit près. CUDA
non-deterministic memory allocation, l'ordre des `del`, la présence d'autres processus sur le
GPU font varier le chiffre. Le carnet en donne la mesure : l'exemple guidé 1 remesure le
**même** chargement FP16 à **7,14 Go**, contre 7,39 Go dans la section 8 -- **3,4 % d'écart**
entre deux mesures du même objet, sur la même machine, dans la même exécution ; et le pic
NF4 de cette section passe au-dessus du FP16 pour les mêmes raisons. Pour les comparaisons
sérieuses (publication, calibration), faire une **moyenne sur plusieurs runs avec
`torch.cuda.empty_cache()` intercalés**.

La cellule qui suit imprime un tableau récapitulatif -- mais attention : **ses valeurs de
mémoire, de qualité et de vitesse sont des ordres de grandeur écrits en dur, pas des mesures
de ce carnet**. Elles viennent du papier QLoRA et du rapport de poids théoriques (16 bits ->
2 octets, 4 bits -> 0,5 octet par paramètre). Les seules mesures de mémoire de cette
exécution sont celles des sections 5 et 8 et de l'exemple guidé 1, citées ci-dessus.

In [20]:
# Tableau recapitulatif complet
print("\n" + "=" * 70)
print("RESUME COMPARATIF : LoRA (FP16) vs QLoRA (4-bit)")
print("=" * 70)
print(f"\n{'Aspect':<30} {'LoRA (FP16)':<20} {'QLoRA (4-bit)':<20}")
print(f"{'-'*30} {'-'*20} {'-'*20}")
print(f"{'Precision poids':<30} {'FP16 (16 bits)':<20} {'NF4 (4 bits)':<20}")
print(f"{'Calculs forward':<30} {'FP16':<20} {'FP16 (dequant)':<20}")
print(f"{'Grad + optimizer':<30} {'FP16 (LoRA seul)':<20} {'FP16 (LoRA seul)':<20}")
print(f"{'VRAM modele ~1.3B':<30} {'~2.6 Go':<20} {'~0.7 Go':<20}")
print(f"{'Qualite vs full FT':<30} {'~98-99%':<20} {'~97-99%':<20}")
print(f"{'Vitesse entrainement':<30} {'1x (ref)':<20} {'~0.8x (dequant)':<20}")
print(f"\nRecommendation :\n"
      "  - Modele < 1B + VRAM suffisante -> LoRA (FP16) : plus simple, plus rapide\n"
      "  - Modele > 1B ou VRAM limitee -> QLoRA : Economie memoire 3-4x")


RESUME COMPARATIF : LoRA (FP16) vs QLoRA (4-bit)

Aspect                         LoRA (FP16)          QLoRA (4-bit)       
------------------------------ -------------------- --------------------
Precision poids                FP16 (16 bits)       NF4 (4 bits)        
Calculs forward                FP16                 FP16 (dequant)      
Grad + optimizer               FP16 (LoRA seul)     FP16 (LoRA seul)    
VRAM modele ~1.3B              ~2.6 Go              ~0.7 Go             
Qualite vs full FT             ~98-99%              ~97-99%             
Vitesse entrainement           1x (ref)             ~0.8x (dequant)     

Recommendation :
  - Modele < 1B + VRAM suffisante -> LoRA (FP16) : plus simple, plus rapide
  - Modele > 1B ou VRAM limitee -> QLoRA : Economie memoire 3-4x


### Lecture du résultat : le triptyque VRAM / qualité / vitesse

Le tableau récapitulatif condense la **promesse et le coût** de QLoRA sur trois axes. Une
précaution d'abord : **ce tableau n'est pas une sortie de mesure** -- ses valeurs de mémoire,
de qualité et de vitesse sont des **ordres de grandeur écrits en dur** (rapport de poids
théoriques, 16 bits -> 2 octets et 4 bits -> 0,5 octet par paramètre, et ordres de grandeur
issus du papier QLoRA pour la qualité et la vitesse). Les mesures réelles de cette exécution
sont dans les sections 5 et 8 et dans l'exemple guidé 1 ; ce tableau sert de **repère de
principe**, pas de chiffre.

Ce que le tableau **énonce** (et qui reste vrai) :

- **VRAM (l'axe décisif)** : `~0,7 Go` (QLoRA) contre `~2,6 Go` (LoRA FP16) pour les
  **poids seuls** d'un modèle de 1,3 Md, soit un rapport d'environ **3,7×**. C'est ce qui rend
  un modèle de 1,3 Md fine-tunable sur une carte *consumer* (8-12 Go) au lieu d'une 24 Go. La
  clé : seuls les poids **NF4 (4 bits)** résident en VRAM ; les gradients et l'optimiseur ne
  portent que sur les **adaptateurs LoRA** (FP16), d'où la ligne `Grad + optimizer : FP16
  (LoRA seul)` identique des deux côtés.
- **Qualité (le prix à payer)** : `~97-99%` du full fine-tuning. La quantization NF4 préserve
  l'information car elle **concentre la précision** là où les poids sont denses (distribution
  gaussienne supposée), mais une dégradation de 1-3 % est intrinsèque à la perte
  d'information 16->4 bits. **C'est un ordre de grandeur, pas une mesure de ce carnet** : la
  seule mesure de qualité disponible ici est la perplexité, et elle porte sur d'autres corpus
  (section 6, exemple guidé 1, exemple guidé 3).
- **Vitesse (le coût caché)** : `~0,8×`, soit ~20 % de ralentissement. La **déquantization**
  NF4->FP16 à chaque *forward pass* a un surcoût de calcul -- QLoRA n'est pas gratuit en temps
  d'entraînement. Ici encore un ordre de grandeur : le carnet ne mesure pas le ratio de
  vitesse entre LoRA et QLoRA.

**Les chiffres réellement mesurés dans cette exécution**, à lire à côté :

| Mesure | Valeur mesurée | Où |
|--------|----------------|-----|
| Empreinte résiduelle du modèle 4 bits (après chargement) | 2,72 Go | section 5 |
| Pic de chargement du même modèle 4 bits | 8,06 Go | section 8 |
| Pic de chargement FP16 | 7,39 Go | section 8 |
| Pic de chargement (NF4 contre FP16) | +9,0 % (surcoût) | section 8 |
| Pic d'entraînement QLoRA (configuration nominale) | 2,13 Go | section 8 |
| Poids FP16 (théorie, 2 octets par paramètre) | 1,63 Go | section 8 |

Trois enseignements que le tableau seul ne donne pas :

1. **Le pic de chargement n'est pas l'empreinte.** Le même modèle 4 bits pèse **2,72 Go** en
   mémoire une fois chargé et **8,06 Go** au pic du chargement : le transitoire de conversion
   représente près de trois fois l'empreinte finale -- il peut même inverser l'ordre avec le
   FP16, comme ici. C'est le pic de chargement, et non l'empreinte, qu'il faut comparer à la
   capacité de la carte.
2. **L'écart à la théorie vient du rapport entre le transitoire et le résiduel**, pas d'une
   « quantification moins efficace qu'annoncé » : sur les poids seuls, NF4 fait bien ce qu'il
   annonce (0,41 Go de théorie pour 1,21 Md de paramètres quantifiés, et 1,03 Go
   d'empreinte de poids mesurés, les embeddings et `lm_head` restant en FP16) ; c'est
   l'ensemble du chargement qui est bruité.
3. **Fine-tuner en QLoRA coûte moins de VRAM que charger seul en FP16** -- 2,13 Go contre
   7,39 Go -- **mais** les deux nombres ne sont pas de même nature : le premier est un pic
   d'entraînement, le second un pic de chargement. La comparaison honnête se fait à
   l'intérieur de chaque catégorie : chargement FP16 7,39 Go contre 4 bits 8,06 Go (le
   transitoire de conversion inverse l'ordre) ; entraînement 4 bits + LoRA 2,13 Go, à
   rapprocher du modèle quantifié en mémoire, 2,72 Go -- deux bases de mesure différentes,
   donc une comparaison à manier avec les mêmes précautions.

**La recommandation** qui clôt le tableau encode la règle de décision : LoRA (FP16) si la
VRAM le permet (plus simple, plus rapide), QLoRA dès que le modèle dépasse ~1 Md de
paramètres ou que la VRAM est contrainte. C'est ce positionnement -- *fine-tuner de gros
modèles sur du petit matériel* -- qui a fait de QLoRA le standard du fine-tuning ouvert sur
GPU *consumer*.

## 9. Nettoyage memoire GPU


**Pourquoi cette cellule compte** : restituer la VRAM (22,4 Go libres après nettoyage) n'est pas de la politesse — la cellule de comparaison précédente a rechargé le modèle deux fois, et les exemples guidés de la section 10 le rechargent pour chaque précision : sans `del` + `empty_cache()`, le cache CUDA saturerait les mesures. Sur un notebook GPU multi-expériences, la discipline mémoire est la condition de mesures honnêtes.


In [21]:
del model_qlora, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"VRAM libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} Go")
else:
    print("Nettoyage termine")

VRAM libre : 1.2 Go


## 10. Exemples guidés et exercices

Cette section applique les concepts du notebook en deux temps : quatre **exemples guidés** résolus et commentés, puis quatre **exercices à compléter** qui mesurent ce que les exemples ne couvrent pas.

Les quatre exemples guidés proviennent d'une contribution étudiante (@nassimJaz, PR #18550) : chaque énoncé, son code de mesure et la lecture du résultat qui suit forment un triplet autonome.

1. **Impact de la précision** — INT8 vs NF4 vs FP16 : VRAM de chargement, empreinte des poids, perplexité et générations.
2. **Rang LoRA et mémoire** — r = 4, 16, 64 : paramètres entraînables, pic VRAM, perte finale, générations.
3. **Double quantization** — `bnb_4bit_use_double_quant` on/off : écart de VRAM mesuré contre la théorie (~0,37 bit/param).
4. **Casser l'invariant d'initialisation** — B initialisé non nul dans le `LoRALinear` de la section 4.

Les exercices à compléter (dernière sous-section) reprennent le flambeau : erreur de reconstruction des poids quantifiés, mémorisation contre généralisation, NF4 contre FP4 à mémoire égale, échelle `alpha/r` contre `alpha/sqrt(r)`. Chaque énoncé porte son critère de réussite mesurable.

### Exemple guidé 1 : impact de la précision (INT8 vs NF4 vs FP16)

*Contribution étudiante de Nassim JAZOULI (@nassimJaz) et Minh NGUYEN (PR #18550), intégrée comme exemple guidé.*

Comparer INT8 (`load_in_8bit=True`), NF4 (4-bit avec double quantization) et FP16 sur le même OPT-1.3B. Pour chaque précision : mesurer la VRAM de chargement (pic), l'empreinte des poids (`get_memory_footprint`), générer en greedy sur un même prompt, et calculer la perplexité sur les 10 extraits d'entraînement comme mesure chiffrée de la dégradation.

**Ce que la théorie annonce** : ~2,6 Go de poids seuls en FP16 (2 octets/paramètre), ~1,3 Go en INT8 (1 octet), ~0,7 Go en NF4 (0,5 octet) ; la VRAM de chargement doit donc se ranger FP16 > INT8 > NF4. La qualité de génération doit se dégrader dans le même ordre, mais faiblement — les trois modèles restent lisibles sur un 1,3B.

Étapes suivies par le corrigé :

1. Définir les trois configurations de chargement (une `torch_dtype`, deux `BitsAndBytesConfig`).
2. Pour chacune : `reset_peak_memory_stats()`, charger, mesurer pic et empreinte, générer, calculer la perplexité sur les extraits, puis libérer la mémoire.
3. Résumer dans un tableau avec l'économie de VRAM relative au FP16.

In [22]:
# Exemple guide 1 : Impact de la precision
# Contribution etudiante de @nassimJaz (PR #18550)
# Comparaison INT8 (load_in_8bit=True) vs NF4 vs FP16 sur OPT-1.3B.
# Pour chaque precision : VRAM de chargement (pic) + generation sur un meme prompt.

# Liberer ce qui reste des sections precedentes (model_4bit est encore reference)
for _name in ["model_4bit", "model_qlora", "trainer", "model_fp16", "model_4b"]:
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

configs_precision = {
    "FP16": dict(torch_dtype=torch.float16),
    "INT8": dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True)),
    "NF4": dict(quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )),
}

prompt_ex1 = "Dans les rues sombres de Paris,"
resultats_ex1 = {}

for label, kwargs in configs_precision.items():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", **kwargs)
    m.eval()
    vram_load = torch.cuda.max_memory_allocated() / 1e9
    empreinte = m.get_memory_footprint() / 1e9

    # Generation greedy (deterministe) pour comparer la qualite a entree egale
    inputs = tokenizer(prompt_ex1, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=40, do_sample=False,
                         repetition_penalty=1.2, pad_token_id=tokenizer.eos_token_id)
    texte = tokenizer.decode(out[0], skip_special_tokens=True)

    # Perplexite sur les textes d'entrainement : mesure chiffree de la degradation
    nll, n_tok = 0.0, 0
    with torch.no_grad():
        for t in training_texts:
            enc = tokenizer(t, return_tensors="pt").to(m.device)
            loss = m(**enc, labels=enc["input_ids"]).loss
            n = enc["input_ids"].shape[1] - 1
            nll += loss.item() * n
            n_tok += n
    ppl = float(torch.exp(torch.tensor(nll / n_tok)))

    resultats_ex1[label] = dict(vram=vram_load, footprint=empreinte, ppl=ppl, texte=texte)
    print(f"\n[{label}] VRAM pic chargement = {vram_load:.2f} Go | empreinte poids = {empreinte:.2f} Go | perplexite = {ppl:.2f}")
    print(f"  -> {texte}")

    del m, inputs, out
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'Precision':<10} {'VRAM pic':>10} {'Empreinte':>10} {'Perplexite':>11} {'Economie VRAM':>14}")
ref = resultats_ex1["FP16"]["vram"]
for label, r in resultats_ex1.items():
    print(f"{label:<10} {r['vram']:>8.2f} G {r['footprint']:>8.2f} G {r['ppl']:>11.2f} {(1 - r['vram'] / ref) * 100:>13.1f}%")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



[FP16] VRAM pic chargement = 7.14 Go | empreinte poids = 2.84 Go | perplexite = 33.89
  -> Dans les rues sombres de Paris, des manifestants ont dénoncé le « traitement » du coronavirus.

« Je ne suis pas un virus ! », a-t-il lancé dans


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



[INT8] VRAM pic chargement = 6.94 Go | empreinte poids = 1.63 Go | perplexite = 33.45
  -> Dans les rues sombres de Paris, des manifestants ont dénoncé la « mauvaise gestion » du coronavirus.

« Je suis triste pour l’économie et je ne pe


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



[NF4] VRAM pic chargement = 7.00 Go | empreinte poids = 1.03 Go | perplexite = 34.48
  -> Dans les rues sombres de Paris, des gens se retrouvent désormais à la recherche du monde.

C'est ce qu'on appelle le "café-déje



Precision    VRAM pic  Empreinte  Perplexite  Economie VRAM
FP16           7.14 G     2.84 G       33.89           0.0%
INT8           6.94 G     1.63 G       33.45           2.8%
NF4            7.00 G     1.03 G       34.48           1.9%


### Lecture du résultat — exemple guidé 1 : la précision achète de la VRAM, pas de la qualité

- **VRAM** : les pics de chargement des trois formats sont ici **presque identiques** — FP16
  7,14 Go, INT8 6,94 Go, NF4 7,00 Go, soit moins de 3 % d'écart et aucun ordre stable.
  C'est la leçon de la section 8 qui se rejoue : le pic de chargement est un **transitoire**
  dominé par les tampons de conversion, il ne discrimine pas les formats sur un 1,3B. Ce qui
  discriminate, c'est l'empreinte des poids (`get_memory_footprint` : **2,84 / 1,63 /
  1,03 Go**), qui suit la théorie (2 octets, 1 octet, 0,5 octet par paramètre quantifié) —
  rapport mesuré de 2,8× entre FP16 et NF4 au lieu des 4× théoriques, l'écart venant des
  couches qui **ne sont pas quantifiées** (les embeddings token + positions, 50 272 × 2048,
  et `lm_head`, laissés en FP16). Sur un 1,3B ces parts fixes pèsent lourd, donc le gain
  relatif est plus faible que sur un 7B.
- **Qualité** : la perplexité mesurée sur les 10 extraits varie d'un peu plus d'un point
  entre les trois formats (33,89 / 33,45 / 34,48). Le classement théorique FP16 ≤ INT8 ≤ NF4
  n'est **pas** confirmé par la mesure : l'INT8 s'affiche même **sous** le FP16, de 0,44
  point -- un écart du même ordre que celui qui sépare le NF4 du FP16 (0,59 point). L'ordre
  des trois formats n'est donc pas établi par cette mesure, et la seule conclusion qui y
  **survit** est que, à cette échelle, la précision n'achète pas de qualité mesurable : elle
  achète de la VRAM. Les générations greedy sont identiques sur les premiers tokens (« Dans
  les rues sombres de Paris, des... »), puis divergent : les trois modèles restent lisibles,
  NF4 conserve l'essentiel de l'information.
- **Remarque** : INT8 (LLM.int8()) est souvent **plus lent** que FP16 à cause de la
  décomposition des outliers, alors que NF4 déquantifie par blocs en FP16. INT8 n'est donc
  pas un compromis évident : pour le fine-tuning mémoire-contraint, NF4 est le meilleur
  choix, ce qui justifie QLoRA.

### Exemple guidé 2 : rang LoRA et mémoire (r = 4, 16, 64)

*Contribution étudiante de Nassim JAZOULI (@nassimJaz) et Minh NGUYEN (PR #18550), intégrée comme exemple guidé.*

Entraîner QLoRA sur OPT-1.3B pour trois rangs — r = 4, 16, 64 — en gardant tout le reste identique. Pour chaque rang : nombre de paramètres entraînables, pic VRAM d'entraînement, perte finale, durée, et une génération sur un prompt fixe (graine fixée). Pour que la comparaison soit équitable, `alpha = 2r` : l'échelle `alpha/r = 2` est la même pour les trois rangs, seule la capacité change.

**Ce que la théorie annonce** : les paramètres entraînables suivent exactement 196 608 × r (formule démontrée en section 4 — la mécanique LoRA), soit 786 432, 3 145 728 et 12 582 912 ; les adaptateurs restent minuscules devant les poids NF4 figés ; la perte finale doit diminuer quand r monte — mais sur 10 exemples vus 15 fois, un rang élevé mémorise plutôt qu'il n'apprend : comparer les générations, pas seulement la perte.

Étapes suivies par le corrigé :

1. Définir une fonction qui charge OPT-1.3B en NF4, greffe un LoRA de rang r (alpha = 2r) et entraîne 15 époques avec les hyperparamètres de la section 6.
2. L'appeler pour r = 4, 16, 64 ; mesurer à chaque fois pic VRAM, perte finale et durée.
3. Comparer dans un tableau : rang, paramètres entraînables (mesurés contre prédits par 196 608 × r), pic VRAM, perte finale, temps.

In [23]:
# Exemple guide 2 : LoRA rank et memoire
# Contribution etudiante de @nassimJaz (PR #18550)
# r = 4, 16, 64 avec QLoRA sur OPT-1.3B : params entrainables, VRAM pic, perte finale.

def train_qlora_rank(r, epochs=15):
    """Charge OPT-1.3B en NF4, greffe LoRA de rang r (alpha = 2r), entraine et mesure."""
    gc.collect()
    torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
    base = prepare_model_for_kbit_training(base)
    cfg = LoraConfig(r=r, lora_alpha=2 * r, target_modules=["q_proj", "v_proj"],
                     lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
    m = get_peft_model(base, cfg)
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)

    args = TrainingArguments(
        output_dir=f"./temp_ft02_r{r}", num_train_epochs=epochs, per_device_train_batch_size=2,
        learning_rate=2e-4, weight_decay=0.01, logging_steps=25, save_strategy="no",
        report_to="none", fp16=True, seed=42, optim="paged_adamw_8bit",
    )
    tr = Trainer(model=m, args=args, train_dataset=tokenized_dataset, data_collator=data_collator)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    res = tr.train()
    duree = time.time() - t0
    pic = torch.cuda.max_memory_allocated() / 1e9

    # Generation echantillonnee (seed fixe) pour comparer qualitativement
    m.eval()
    torch.manual_seed(0)
    inputs = tokenizer("Jean Valjean regardait la nuit", return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.8, top_p=0.9,
                         pad_token_id=tokenizer.eos_token_id)
    texte = tokenizer.decode(out[0], skip_special_tokens=True)

    del m, base, tr, inputs, out
    gc.collect()
    torch.cuda.empty_cache()
    return dict(r=r, trainable=n_train, predit=196_608 * r, vram_peak=pic,
                loss=res.training_loss, temps=duree, texte=texte)


resultats_ex2 = []
for r in [4, 16, 64]:
    print(f"\n===== r = {r} =====")
    res = train_qlora_rank(r)
    resultats_ex2.append(res)
    print(f"generation : {res['texte']}")

print(f"\n{'r':>3} {'params entrainables':>20} {'predit 196608*r':>16} {'VRAM pic':>10} {'perte finale':>13} {'temps':>8}")
for res in resultats_ex2:
    print(f"{res['r']:>3} {res['trainable']:>20,} {res['predit']:>16,} {res['vram_peak']:>8.2f} G "
          f"{res['loss']:>13.4f} {res['temps']:>6.1f} s")


===== r = 4 =====


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,3.302095
50,2.742125
75,2.488779


generation : Jean Valjean regardait la nuit. Les garçons de la prison ne pouvaient que dormir. Les garçons de la prison ne pouvaient que dormir. Les garçons de la prison ne pouvaient que dormir. Les

===== r = 16 =====


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,2.757869
50,1.699724
75,1.214468


generation : Jean Valjean regardait la nuit dans l'espoir de s'enjouir. Le houleur d'un mouvement intolérable n'était pas sûr. Cosette ne voyait ni le bien ni

  r  params entrainables  predit 196608*r   VRAM pic  perte finale    temps
  4              786,432          786,432     6.40 G        2.8443   34.6 s
 16            3,145,728        3,145,728     6.84 G        2.3948   32.2 s
 64           12,582,912       12,582,912     7.30 G        1.8907   38.1 s


### Lecture du résultat — exemple guidé 2 : le rang achète de la capacité, pas un meilleur modèle

- **Paramètres entraînables** : ils suivent exactement la formule `196 608 × r` démontrée en
  section 4 (786 432 / 3 145 728 / 12 582 912) -- la colonne « prédit » coïncide avec le
  comptage réel.
- **VRAM pic** : elle grimpe de **6,40 Go** (r=4) à **6,84 Go** (r=16) puis **7,30 Go**
  (r=64), soit 0,90 Go d'écart sur les trois rangs. Les adaptateurs seuls restent minuscules
  (~12,6 M paramètres à r=64, soit ~25 Mo de poids FP16) : la différence vient des
  gradients, des états de l'optimiseur paginé et des tampons intermédiaires qui accompagnent
  chaque paramètre entraînable -- un pic VRAM se mesure, il ne se déduit pas d'un comptage de
  paramètres. Les poids NF4 figés et les activations (séquences de 256 tokens) dominent
  toujours la mémoire, mais à r=64 l'adaptateur n'est plus gratuit. Ces pics sont mesurés
  dans le contexte propre à cet exemple (chaque rang recharge sa base) : ils ne sont pas
  directement comparables au pic d'entraînement de la section 8 (2,13 Go), mesuré sur le
  modèle déjà chargé et quantifié.
- **Perte finale** : elle diminue quand r augmente (2,8443 -> 2,3948 -> 1,8907), mais avec 10
  exemples vus 15 fois, un rang élevé sert surtout à **mémoriser** le corpus. Les générations
  le montrent : à r=4, le modèle boucle sur une phrase courte et interchangeable (« Les
  garçons de la prison ne pouvaient que dormir », répétée telle quelle) ; à r=16, il
  s'aventure dans des tournures plus construites mais fautives (« Ce que l'homme de la loi
  n'ait jamais vu dans les années suivantes ») ; à r=64, le vocabulaire devient ambitieux et
  se casse (« Le houleur d'un mouvement intolérable n'était pas sûr ») -- la perte baisse, la
  grammaire ne suit pas. Une perte plus basse n'est donc pas synonyme d'un meilleur modèle :
  sans jeu de validation, on ne mesure que la mémorisation -- c'est exactement la question
  que le balayage multi-graines étendu ci-dessous retourne sur quatre leviers. Pour ce
  dataset, r=8-16 reste le compromis raisonnable.
- **Coût** : 34,6 s à r=4, 32,2 s à r=16, 38,1 s à r=64 : à cette échelle le temps de calcul
  ne suit le rang que grossièrement -- seule la VRAM le suit nettement. Le rang élevé se
  paie surtout en mémoire.
- Pour que la comparaison soit équitable, `alpha = 2r` : l'échelle `alpha/r = 2` est
  identique pour les trois rangs, seule la capacité change.

### Exemple guidé 3 : double quantization

*Contribution étudiante de Nassim JAZOULI (@nassimJaz) et Minh NGUYEN (PR #18550), intégrée comme exemple guidé.*

La double quantization (`bnb_4bit_use_double_quant=True`) compresse les constantes de quantization elles-mêmes en FP8, économisant ~0,37 bit/param. Comparer avec et sans pour mesurer l'impact réel sur la VRAM et la qualité.

**Protocole** : mesurer la VRAM de chargement dans les deux configurations avec `reset_peak_memory_stats()` avant chaque `from_pretrained` (protocole de la section 3). L'écart attendu — ~56 Mo pour les ~1,21 Md de paramètres effectivement quantifiés d'OPT-1.3B (× 0,373 bit ÷ 8) — est petit devant le bruit des pics de chargement : exécuter chaque mesure 3 fois et comparer les minima, pas les pics uniques. Un écart nul ne veut pas dire « la double quantization ne marche pas » : vérifier d'abord que les deux `BitsAndBytesConfig` ne diffèrent bien que par `bnb_4bit_use_double_quant`.

In [24]:
# Exemple guide 3 : Double quantization on/off
# Contribution etudiante de @nassimJaz (PR #18550)
# Protocole : reset_peak_memory_stats avant chaque from_pretrained, 3 repetitions, on garde les minima.

def charger_nf4(double_quant):
    cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=double_quant,   # SEULE difference entre les deux configs
    )
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=cfg, device_map="auto")


resultats_ex3 = {}
for dq in [False, True]:
    pics, allocs = [], []
    for _ in range(3):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        m = charger_nf4(dq)
        pics.append(torch.cuda.max_memory_allocated() / 1e9)
        allocs.append(torch.cuda.memory_allocated() / 1e9)
        footprint = m.get_memory_footprint() / 1e9
        del m
    # Qualite : perplexite sur les extraits (un chargement supplementaire)
    m = charger_nf4(dq)
    m.eval()
    nll, n_tok = 0.0, 0
    with torch.no_grad():
        for t in training_texts:
            enc = tokenizer(t, return_tensors="pt").to(m.device)
            n = enc["input_ids"].shape[1] - 1
            nll += m(**enc, labels=enc["input_ids"]).loss.item() * n
            n_tok += n
    ppl = float(torch.exp(torch.tensor(nll / n_tok)))
    del m, enc
    gc.collect()
    torch.cuda.empty_cache()
    resultats_ex3[dq] = dict(pic=min(pics), alloc=min(allocs), footprint=footprint, ppl=ppl)

print(f"{'double_quant':<14} {'pic min':>9} {'alloue min':>11} {'empreinte':>10} {'perplexite':>11}")
for dq, r in resultats_ex3.items():
    print(f"{str(dq):<14} {r['pic']:>7.3f} G {r['alloc']:>9.3f} G {r['footprint']:>8.3f} G {r['ppl']:>11.3f}")

ecart_mo = (resultats_ex3[False]["alloc"] - resultats_ex3[True]["alloc"]) * 1e3
print(f"\nEconomie mesuree (memoire allouee) : {ecart_mo:.1f} Mo")

# Theorie : seules les couches lineaires des blocs sont quantifiees (pas les embeddings / lm_head)
n_quant = 24 * (4 * 2048 * 2048 + 2 * 2048 * 8192)
print(f"Params quantifies : {n_quant:,} -> theorie 0.373 bit/param = {n_quant * 0.373 / 8 / 1e6:.1f} Mo")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


... (7 outputs left out)

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


double_quant     pic min  alloue min  empreinte  perplexite
False            8.154 G     6.692 G    1.025 G      34.450
True             8.290 G     6.636 G    1.025 G      34.480

Economie mesuree (memoire allouee) : 56.1 Mo
Params quantifies : 1,207,959,552 -> theorie 0.373 bit/param = 56.3 Mo


### Lecture du résultat — exemple guidé 3 : un gain minuscule, linéaire dans la taille du modèle

- **Théorie** : sans double quantization, chaque bloc de 64 poids a une constante d'échelle
  en FP32 -> 32/64 = 0,5 bit/param. Avec double quantization, ces constantes sont stockées en
  8 bits par blocs de 256, plus une constante FP32 par bloc de 256 constantes -> 8/64 +
  32/(64·256) ≈ 0,127 bit/param. Gain : **≈ 0,373 bit/param**.
- **Mesure** : seules les couches linéaires des 24 blocs de décodeur sont quantifiées
  (1 207 959 552 paramètres, soit ≈ 1,21 Md ; les embeddings et `lm_head` restent en FP16).
  L'économie attendue est donc ≈ 56 Mo -- et l'écart mesuré sur la mémoire allouée tombe
  dessus : **56,1 Mo mesurés** contre **56,3 Mo prédits** par la théorie. Sur le **pic** de
  chargement, la même comparaison (8,290 contre 8,154 Go) est plus bruitée encore -- le
  signe même de l'écart y change -- car le pic contient des tampons temporaires de
  conversion : d'où l'intérêt de répéter la mesure et de comparer les minima.
- **Qualité** : la perplexité est pratiquement identique entre les deux configurations
  (34,450 contre 34,480 -- 0,030 point, de l'ordre du bruit numérique) : les constantes
  d'échelle quantifiées en 8 bits sont assez précises pour ne rien changer au résultat.
- **Conclusion** : sur OPT-1.3B le gain est marginal -- 56 Mo sur une mémoire allouée de
  6,7 Go, soit moins de 1 % -- mais il grandit **linéairement** avec la taille : ~ 0,3 Go
  pour un 7B, ~ 3 Go pour un 65B -- c'est ce qui permettait dans le papier QLoRA de faire
  tenir un 65B sur un seul GPU de 48 Go. C'est un gain « gratuit », d'où son activation par
  défaut.

### Exemple guidé 4 : casser l'invariant d'initialisation

*Contribution étudiante de Nassim JAZOULI (@nassimJaz) et Minh NGUYEN (PR #18550), intégrée comme exemple guidé.*

L'invariant de la section 4 : `LoRALinear` initialise `B` à zéro, donc `ΔW = B·A = 0` et le modèle fine-tunable démarre exactement comme le pré-entraîné. Cet exemple casse exprès cet invariant : initialiser `B` à `normal(0, b_std)` au lieu de zéro, et observer ce que ce choix détruit.

Mesures :

1. Pour b_std = 0, 0.001, 0.01, 0.1 sur une couche `nn.Linear` 512×512 identique à celle de la section 4 : norme maximale de `ΔW` initial, rapport `|s·ΔW| / |W₀|`, test `torch.allclose(lora(x), base(x))` avant tout entraînement, écart maximal de sortie.
2. Reprendre la régression de la section 4 (50 pas SGD, même learning rate) avec B = 0 puis B ≠ 0 : comparer les pertes initiales et finales.

**Indice de raisonnement** : reprendre la classe de la section 4 et remplacer uniquement l'initialisation de B ; si A et B étaient tous deux nuls, les gradients seraient nuls et rien n'apprendrait jamais.

In [25]:
# Exemple guide 4 : initialiser B NON nul et observer le démarrage
# Contribution etudiante de @nassimJaz (PR #18550)
def init_b_nonzero(base_layer, r=8, lora_alpha=16, b_std=0.01):
    """
    Retourner un LoRALinear dont B est initialise a normal(0, b_std) au lieu de zero.
    Mesurer : delta_W initial, forward == base ?, et interpreter pourquoi
    le modele LoRA ne demarre PLUS exactement comme le pre-entraine.
    """
    lora = LoRALinear(base_layer, r=r, lora_alpha=lora_alpha)
    with torch.no_grad():
        lora.lora_B.normal_(0, b_std)       # B != 0  ->  delta_W = B @ A != 0
    return lora


# Couche de base fraiche (identique a la section 4), pour ne pas dependre de lora_demo
torch.manual_seed(42)
base_ex4 = nn.Linear(512, 512, bias=False)
with torch.no_grad():
    base_ex4.weight.copy_(torch.randn(512, 512) * 0.02)

x_probe4 = torch.randn(4, 512)
print(f"{'b_std':>7} {'max|dW|':>10} {'|s*dW|/|W0|':>12} {'forward==base':>14} {'ecart sortie max':>17}")
for b_std in [0.0, 0.001, 0.01, 0.1]:
    torch.manual_seed(0)
    lora_nz = init_b_nonzero(base_ex4, r=8, lora_alpha=16, b_std=b_std)
    dW = lora_nz.delta_W().detach()
    with torch.no_grad():
        out_nz, out_b = lora_nz(x_probe4), base_ex4(x_probe4)
    rel = (lora_nz.scaling * dW).norm() / base_ex4.weight.norm()
    print(f"{b_std:>7} {dW.abs().max():>10.2e} {rel:>12.2%} {str(torch.allclose(out_nz, out_b)):>14} "
          f"{(out_nz - out_b).abs().max():>17.2e}")

# Consequence sur l'apprentissage : meme tache, meme lr, B=0 vs B!=0
print("\nMeme regression que la section 4 (50 pas SGD, lr=0.1) :")
torch.manual_seed(1)
xb, tgt = torch.randn(16, 512), torch.randn(16, 512)
for b_std in [0.0, 0.1]:
    torch.manual_seed(0)
    l = init_b_nonzero(base_ex4, b_std=b_std)
    opt4 = torch.optim.SGD([l.lora_A, l.lora_B], lr=0.1)
    losses = []
    for _ in range(50):
        opt4.zero_grad()
        loss = ((l(xb) - tgt) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        opt4.step()
    print(f"  b_std={b_std:<4} : loss initiale = {losses[0]:.4f} -> finale = {losses[-1]:.4f}")

  b_std    max|dW|  |s*dW|/|W0|  forward==base  ecart sortie max
    0.0   0.00e+00        0.00%           True          0.00e+00
  0.001   4.38e-04        0.73%          False          1.07e-02
   0.01   4.38e-03        7.25%          False          1.07e-01


    0.1   4.38e-02       72.50%          False          1.07e+00

Meme regression que la section 4 (50 pas SGD, lr=0.1) :
  b_std=0.0  : loss initiale = 1.2279 -> finale = 1.1487
  b_std=0.1  : loss initiale = 1.3386 -> finale = 1.2085


### Lecture du résultat — exemple guidé 4 : pourquoi B doit naître nul

- Avec `B = 0` (b_std = 0), on retrouve l'invariant : `ΔW = 0`, `forward == base → True`,
  écart de sortie **0,00**.
- Dès que `B ≠ 0`, `ΔW = B·A` est une matrice aléatoire non nulle de rang ≤ r, et
  `forward == base → False` : **avant le moindre pas d'entraînement**, le modèle n'est plus
  le pré-entraîné. Il est perturbé par un bruit `(α/r)·B·A·x` dont l'amplitude croît avec
  `b_std` ; la colonne `|s·ΔW| / |W₀|` le mesure : **0,73 %** à b_std = 0,001, **7,25 %** à
  0,01, **72,50 %** à 0,1. L'écart de sortie maximal suit la même progression (1,07e-02,
  1,07e-01, 1,07e+00) : le bruit injecté est proportionnel à `b_std`, comme le veut
  `ΔW = B·A`.
- **Pourquoi c'est un problème** : sur un vrai LLM, ce bruit est injecté dans *toutes* les
  couches ciblées (48 projections pour OPT-1.3B) et se propage à travers le réseau : la perte
  de départ est plus élevée, et les premiers pas d'optimisation servent à « désapprendre » ce
  bruit au lieu d'apprendre la tâche. Avec un `b_std` grand, on peut même détruire les
  capacités du modèle (sorties incohérentes) dès l'initialisation. On le voit sur la
  régression jouet : la loss initiale est plus haute avec B ≠ 0 (**1,3386** contre
  **1,2279** pour b_std = 0), et elle reste plus haute à la fin de l'entraînement (1,2085
  contre 1,1487).
- **Pourquoi A aléatoire et B nul (et pas l'inverse, ni les deux à zéro)** : si A et B étaient
  tous deux nuls, les gradients `∂L/∂A` et `∂L/∂B` seraient nuls : rien n'apprendrait jamais.
  Avec A aléatoire et B = 0, le produit vaut 0 (démarrage exact) mais `∂L/∂B` est non nul dès
  le premier pas, donc l'apprentissage démarre. C'est le seul choix qui combine **démarrage
  exact** et **gradient non nul**.


### Balayage multi-graines étendu : le rang, le rapport alpha/r, les modules et les époques

L'exemple guidé 2 a mesuré le rang sur **une seule graine**, et sur la perte d'entraînement. Une variation sur une graine ne dit rien : le bruit d'entraînement peut produire l'écart à lui seul. Ce qui suit rejoue chaque configuration sur les trois graines `0`, `7` et `42`, et mesure la **perplexité retenue** — celle qui distingue mémoriser de généraliser.

Quatre leviers sont balayés, un à la fois :

| Levier | Valeurs | Ce qui est tenu constant, et pourquoi |
|---|---|---|
| **Rang** `r` | 4, 16, 64 | l'échelle `alpha / r = 2` : sinon deux grandeurs changent ensemble et l'écart n'est attribuable à aucune des deux |
| **Rapport** `alpha / r` | 0,5 · 1 · 2 · 4 | le rang `r = 16` : le rapport ne peut être isolé qu'à capacité constante |
| **Modules ciblés** | `q_proj` + `v_proj` ; les quatre projections `q, k, v, o` | le rang, le taux, le rapport, les époques |
| **Époques** | 15, perplexité retenue mesurée **à chaque époque** | la configuration nominale du carnet |

La règle de lecture est la même que pour le taux d'apprentissage : la colonne qui décide est l'**écart-type entre graines**. Un écart entre deux configurations qui ne dépasse pas cette dispersion est du bruit, et le carnet doit l'écrire ainsi.

Le dernier levier ne produit pas un tableau mais une **courbe** : c'est lui qui montre le moment où la perplexité retenue cesse de baisser alors que la perte d'entraînement continue de descendre. C'est ce que l'arrêt précoce exploite, et c'est l'objet du nouvel exercice 2.

La grille complète fait **30 entraînements** ; sur le GPU de la machine de référence, chacun coûte quelques dizaines de secondes, la section entière une quinzaine de minutes. C'est le prix d'un balayage qui distingue un effet d'une fluctuation — et c'est exactement ce qu'une graine unique ne peut pas acheter.

In [26]:
# --- Balayage multi-graines : le rang LoRA, a echelle alpha/r constante ---
# `entrainer_et_evaluer` est reutilisee telle quelle : meme protocole, meme
# corpus retenu, memes graines. Seuls le rang et alpha changent, ensemble,
# pour que alpha/r = 2 reste vrai a chaque rang.
RANGS = [4, 16, 64]

resultats_rang = []
for rang in RANGS:
    for graine in GRAINES:
        cfg = dict(CONFIG, lora_r=rang, lora_alpha=2 * rang, seed=graine)
        res_rang = entrainer_et_evaluer(cfg)
        resultats_rang.append(res_rang)
        print(f"r={rang:<3} graine={graine:<3} perte={res_rang['perte']:.4f}  "
              f"ppl {res_rang['ppl_avant']:.3f} -> {res_rang['ppl_apres']:.3f} "
              f"({res_rang['delta']:+.3f})  vram={res_rang['vram_peak_go']:.2f} Go  "
              f"{res_rang['duree_s']:.0f}s")


def synthese(resultats, cle, valeurs):
    """Tableau de synthese : moyenne ET dispersion, pour chaque valeur de `cle`."""
    print(f"{'config':>10} {'perte (moy)':>12} {'perte (ect)':>12} "
          f"{'ppl apres (moy)':>16} {'ppl (ect)':>10} {'variation (moy)':>16}")
    for valeur in valeurs:
        lignes = [r for r in resultats if r[cle] == valeur]
        pertes = [r["perte"] for r in lignes]
        ppls = [r["ppl_apres"] for r in lignes]
        moy_p = sum(pertes) / len(pertes)
        ect_p = (sum((x - moy_p) ** 2 for x in pertes) / len(pertes)) ** 0.5
        moy_q = sum(ppls) / len(ppls)
        ect_q = (sum((x - moy_q) ** 2 for x in ppls) / len(ppls)) ** 0.5
        moy_v = sum(r["delta"] for r in lignes) / len(lignes)
        print(f"{str(valeur):>10} {moy_p:>12.4f} {ect_p:>12.4f} {moy_q:>16.3f} "
              f"{ect_q:>10.3f} {moy_v:>+16.3f}")


print()
print("=" * 78)
print("SYNTHESE -- le rang LoRA (3 graines par rang)")
print("=" * 78)
synthese(resultats_rang, "lora_r", RANGS)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,3.307113
50,2.767329
75,2.486399


r=4   graine=0   perte=2.8536  ppl 31.598 -> 24.375 (-7.223)  vram=7.69 Go  33s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


... (26 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,2.746104
50,1.691853
75,1.206883


r=64  graine=42  perte=1.8816  ppl 31.598 -> 34.422 (+2.824)  vram=11.16 Go  174s

SYNTHESE -- le rang LoRA (3 graines par rang)
    config  perte (moy)  perte (ect)  ppl apres (moy)  ppl (ect)  variation (moy)
         4       2.8595       0.0042           24.964      0.567           -6.634
        16       2.3979       0.0098           28.910      0.510           -2.688
        64       1.8745       0.0107           34.383      0.265           +2.785


### Lecture du résultat — le rang achète de la mémorisation, pas de la généralisation

Sur une seule graine et sur la seule perte d'entraînement, l'exemple guidé 2 concluait que la perte baisse quand le rang monte. Les trois graines et la perplexité retenue renversent la moitié de cette conclusion :

- la **perte d'entraînement** descend bien avec le rang (2,860 → 2,398 → 1,875) ;
- la **perplexité retenue** remonte au contraire (24,964 → 28,910 → 34,383) : à `r = 64`, l'adaptateur est *pire que la base* (+2,785) sur des textes qu'il n'a jamais vus.

L'effet est réel, pas un bruit de graine : l'écart entre deux rangs adjacents va de 4 à 5 points de perplexité, quand la dispersion entre graines reste entre 0,265 et 0,567 — un ordre de grandeur d'écart, dans le même sens sur les trois graines. Les deux colonnes disent ensemble que le rang supplémentaire sert à mémoriser les dix extraits, et que cette mémorisation dégrade le reste : la conclusion de l'exemple guidé 2 (« le rang achète de la capacité ») reste vraie en capacité, fausse en généralisation — sur ce corpus, le petit rang gagne.

La colonne « perte (écart-type) » mérite d'être lue aussi : elle est minuscule (0,004 à 0,011) devant les écarts de perte entre rangs (0,46 à 0,52). Autrement dit, le rang est un levier dont l'effet **dépasse massivement** le bruit d'entraînement — c'est exactement la question que la graine unique de l'exemple guidé 2 ne pouvait pas trancher.

Le balayage du rang a tenu `alpha / r = 2` constant, ce qui est la bonne façon de mesurer l'effet du rang seul — mais cela laisse une question ouverte : lequel des deux réglages porte l'effet ? Le rang fixe la **capacité** de l'adaptateur ; `alpha` fixe l'**échelle** de sa mise à jour, et c'est leur rapport qui décide de la force du signal en retour.

Pour séparer les deux, le rang est figé (r = 16, le choix nominal du carnet) et seul `alpha` varie, ce qui fait prendre au rapport `alpha / r` les valeurs **0,5 · 1 · 2 · 4**. Le point 2 est la configuration déjà mesurée plus haut — il sert de repère, et retrouver la même valeur est le contrôle que le protocole est bien le même.

In [27]:
# --- Balayage multi-graines : le rapport alpha/r, a rang fige ---
# r = 16 constant ; alpha varie, donc alpha/r prend les valeurs 0.5, 1, 2, 4.
RATIOS_ALPHA = [0.5, 1, 2, 4]
RANG_FIGE = 16

resultats_ratio = []
for ratio in RATIOS_ALPHA:
    for graine in GRAINES:
        cfg = dict(CONFIG, lora_r=RANG_FIGE,
                   lora_alpha=int(ratio * RANG_FIGE), seed=graine)
        res_ratio = entrainer_et_evaluer(cfg)
        res_ratio["ratio"] = ratio
        resultats_ratio.append(res_ratio)
        print(f"alpha/r={ratio:<4} graine={graine:<3} perte={res_ratio['perte']:.4f}  "
              f"ppl {res_ratio['ppl_avant']:.3f} -> {res_ratio['ppl_apres']:.3f} "
              f"({res_ratio['delta']:+.3f})  vram={res_ratio['vram_peak_go']:.2f} Go  "
              f"{res_ratio['duree_s']:.0f}s")

print()
print("=" * 78)
print("SYNTHESE -- le rapport alpha/r (rang 16, 3 graines par rapport)")
print("=" * 78)
synthese(resultats_ratio, "ratio", RATIOS_ALPHA)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,3.300651
50,2.766426
75,2.496413


alpha/r=0.5  graine=0   perte=2.8545  ppl 31.598 -> 24.837 (-6.761)  vram=11.56 Go  95s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


... (38 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,2.882829
50,2.007797
75,1.622353


alpha/r=4    graine=42  perte=2.1710  ppl 31.598 -> 31.741 (+0.142)  vram=16.27 Go  282s

SYNTHESE -- le rapport alpha/r (rang 16, 3 graines par rapport)
    config  perte (moy)  perte (ect)  ppl apres (moy)  ppl (ect)  variation (moy)
       0.5       2.8723       0.0130           25.063      0.283           -6.536
         1       2.6258       0.0140           26.635      0.416           -4.964
         2       2.3958       0.0121           28.992      0.500           -2.607
         4       2.1556       0.0127           30.808      0.663           -0.791


### Lecture du résultat — plus l'échelle `alpha/r` est petite, mieux le corpus retenu

À rang constant, la réponse est **monotone** : plus `alpha/r` grandit, plus la perte d'entraînement descend (2,872 → 2,626 → 2,396 → 2,156) et plus la perplexité retenue se dégrade (25,063 → 26,635 → 28,992 → 30,808). La variation passe de −6,536 à −0,791 : à `alpha/r = 4`, l'adaptateur n'améliore presque plus les textes qu'il n'a pas vus, alors qu'il mémorise mieux que jamais.

Chaque écart entre deux rapports adjacents (1,6 à 2,4 points de perplexité) dépasse la dispersion entre graines (0,283 à 0,663) : l'effet de l'échelle est réel, lui aussi.

**Le contrôle interne tombe juste** : la ligne `alpha/r = 2` de ce balayage et la ligne `r = 16` du balayage du rang décrivent la même configuration (rang 16, échelle 2) mesurée deux fois, à six configurations d'intervalle. Elles rendent 28,992 et 28,910 — écart de 0,08, inférieur à la dispersion de chacune. Le protocole est répétable à l'intérieur du même run ; c'est ce qui autorise à lire les quatre tableaux de la section comme une seule expérience.

Le rang et l'échelle racontent donc **la même histoire par deux chemins** : restreindre la mise à jour (peu de paramètres, ou petite échelle) préserve la généralisation sur ce petit corpus ; l'amplifier (grand rang, grande échelle) achète de la mémorisation au prix des textes retenus. Les deux leviers agissent sur la même grandeur — la norme effective de la correction LoRA — et c'est pourquoi leurs courbes se ressemblent autant.

### Où greffer les adaptateurs ? La question posée avant la mesure

Le rang règle **combien** de paramètres l'adaptateur ajoute. Les modules ciblés règlent **où** il les ajoute. `q_proj` et `v_proj` est le choix minimal du papier LoRA ; les quatre projections `q, k, v, o` touchent deux fois plus de couches à rang égal.

La question est la même que pour le rang et le rapport `alpha / r` : à taux, rang et époques constants, la perplexité retenue bouge-t-elle plus que la dispersion entre graines ? Si oui, « où greffer » compte autant que « combien ». Si non, le choix des modules est une affaire de coût et de surface mémoire, pas de qualité — et le balayage ci-dessous le tranche.

In [28]:
# --- Balayage multi-graines : quelles projections porter avec LoRA ? ---
# Meme rang, meme taux, memes graines : seule la liste des projections change.
MODULES_CIBLES = {
    "q+v": ["q_proj", "v_proj"],
    "q+k+v+o": ["q_proj", "k_proj", "v_proj", "o_proj"],
}

resultats_modules = []
for nom_modules, modules in MODULES_CIBLES.items():
    for graine in GRAINES:
        cfg = dict(CONFIG, target_modules=modules, seed=graine)
        res_mod = entrainer_et_evaluer(cfg)
        res_mod["modules"] = nom_modules
        resultats_modules.append(res_mod)
        print(f"{nom_modules:<8} graine={graine:<3} perte={res_mod['perte']:.4f}  "
              f"ppl {res_mod['ppl_avant']:.3f} -> {res_mod['ppl_apres']:.3f} "
              f"({res_mod['delta']:+.3f})  vram={res_mod['vram_peak_go']:.2f} Go  "
              f"{res_mod['duree_s']:.0f}s")

print()
print("=" * 78)
print("SYNTHESE -- les modules cibles (3 graines par configuration)")
print("=" * 78)
synthese(resultats_modules, "modules", list(MODULES_CIBLES))

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,2.970593
50,2.245178
75,1.925953


q+v      graine=0   perte=2.3806  ppl 31.598 -> 27.647 (-3.951)  vram=16.70 Go  335s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


... (14 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,2.959751
50,2.145843
75,1.772335


q+k+v+o  graine=42  perte=2.2926  ppl 31.598 -> 28.229 (-3.369)  vram=18.85 Go  413s

SYNTHESE -- les modules cibles (3 graines par configuration)
    config  perte (moy)  perte (ect)  ppl apres (moy)  ppl (ect)  variation (moy)
       q+v       2.3963       0.0125           28.663      0.724           -2.936
   q+k+v+o       2.3009       0.0102           28.768      0.631           -2.830


### Lecture du résultat — où greffer : indiscernable du bruit

Les quatre projections font mieux que deux sur la perte d'entraînement (2,301 contre 2,396 — un écart de 0,095, pour des dispersions de l'ordre de 0,01) : à rang égal, toucher `k_proj` et `o_proj` en plus donne réellement plus de capacité à mémoriser. Mais la perplexité retenue ne suit pas : 28,768 contre 28,663, un écart de **0,105 point**, six fois plus petit que la dispersion entre graines (0,631 à 0,724). Sur ce corpus, le choix des modules ciblés ne se distingue pas du bruit : il se règle sur le coût — temps et VRAM, mesurés dans les colonnes du balayage — pas sur la qualité.

C'est la contre-épreuve de la section. Un levier réel (le rang, l'échelle, le taux d'apprentissage) produit des écarts de 1,6 à 5 points de perplexité retenue ; un levier inopérant ici en produit 0,1. La même règle de lecture — « l'écart dépasse-t-il la dispersion entre graines ? » — rend les deux verdicts, et c'est précisément pourquoi elle est appliquée systématiquement au lieu de commenter les moyennes seules.

Les trois balayages précédents choisissent une **configuration** et la mènent à son terme : quinze époques, toujours. Or rien ne garantit que l'époque 15 soit la meilleure. Sur dix extraits vus quinze fois, la perplexité retenue peut avoir atteint son minimum bien avant, puis remonter — l'adaptateur commence alors à mémoriser au lieu de styliser.

Pour le voir, il faut mesurer **pendant** l'entraînement, pas seulement à la fin. La sonde ci-dessous est un `TrainerCallback` : à chaque fin d'époque, elle mesure la perplexité sur le même corpus retenu que le reste du carnet, et l'empile. La courbe qui en sort se lit toujours avec la dispersion entre graines, parce que le minimum d'une graine n'est pas forcément celui d'une autre.

C'est cette courbe qui rend l'arrêt précoce possible : au lieu de fixer les époques, on arrête quand la perplexité retenue ne s'améliore plus, et on garde l'adaptateur de la meilleure époque.

In [29]:
# --- Perplexite retenue epoque par epoque, sur les trois graines ---
# La question n'est plus "quelle configuration gagne" mais "a quel moment
# l'adaptateur cesse de generaliser". On mesure APRES CHAQUE epoque.
from transformers import TrainerCallback


class SondeRetenue(TrainerCallback):
    """Mesure la perplexite retenue a la fin de chaque epoque."""

    def __init__(self, modele, textes):
        self.modele = modele
        self.textes = textes
        self.courbe = []

    def on_epoch_end(self, args, state, control, **kwargs):
        self.modele.eval()
        self.courbe.append((round(state.epoch), perplexite(self.modele, self.textes)))
        self.modele.train()
        return control


def entrainer_avec_courbe(config, textes_retenus):
    """Entraine et rend la perplexite retenue a chaque epoque."""
    gc.collect()
    torch.cuda.empty_cache()

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
    base = prepare_model_for_kbit_training(base)
    ppl_avant = perplexite(base, textes_retenus)

    lora = LoraConfig(
        r=config["lora_r"], lora_alpha=config["lora_alpha"],
        target_modules=config["target_modules"], lora_dropout=config["lora_dropout"],
        bias="none", task_type="CAUSAL_LM")
    modele = get_peft_model(base, lora)

    sonde = SondeRetenue(modele, textes_retenus)
    args = TrainingArguments(
        output_dir=tempfile.mkdtemp(prefix=f"ft02_courbe_s{config['seed']}_"),
        num_train_epochs=config["num_train_epochs"],
        per_device_train_batch_size=config["batch_size"],
        learning_rate=config["learning_rate"], weight_decay=0.01,
        logging_steps=25, save_strategy="no", report_to="none",
        fp16=torch.cuda.is_available(), seed=config["seed"],
        optim="paged_adamw_8bit",
    )
    trainer = Trainer(model=modele, args=args, train_dataset=tokenized_dataset,
                      data_collator=data_collator, callbacks=[sonde])

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    res = trainer.train()
    duree = time.time() - t0

    del modele, base, trainer
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(args.output_dir, ignore_errors=True)
    return {"seed": config["seed"], "ppl_avant": ppl_avant,
            "perte": res.training_loss, "courbe": sonde.courbe, "duree_s": duree}


courbes = []
for graine in GRAINES:
    res_courbe = entrainer_avec_courbe(dict(CONFIG, seed=graine), heldout_texts)
    courbes.append(res_courbe)
    epoque_min, ppl_min = min(res_courbe["courbe"], key=lambda point: point[1])
    print(f"graine={graine:<3} ppl avant={res_courbe['ppl_avant']:.3f}  "
          f"meilleure epoque={epoque_min:>2} (ppl {ppl_min:.3f})  "
          f"ppl a la derniere epoque={res_courbe['courbe'][-1][1]:.3f}  "
          f"{res_courbe['duree_s']:.0f}s")

print()
print("=" * 78)
print("COURBE -- perplexite retenue par epoque (moyenne et ecart-type sur 3 graines)")
print("=" * 78)
print(f"{'epoque':>7} {'ppl (moy)':>12} {'ppl (ect)':>12}")
for i in range(CONFIG["num_train_epochs"]):
    valeurs = [r["courbe"][i][1] for r in courbes if len(r["courbe"]) > i]
    if not valeurs:
        continue
    moy = sum(valeurs) / len(valeurs)
    ect = (sum((v - moy) ** 2 for v in valeurs) / len(valeurs)) ** 0.5
    print(f"{i + 1:>7} {moy:>12.3f} {ect:>12.3f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,2.969499
50,2.250158
75,1.933804


graine=0   ppl avant=31.598  meilleure epoque= 2 (ppl 24.434)  ppl a la derniere epoque=28.119  422s


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


... (2 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Step,Training Loss
25,3.028861
50,2.253954
75,1.943491


graine=42  ppl avant=31.598  meilleure epoque= 3 (ppl 24.264)  ppl a la derniere epoque=29.137  384s

COURBE -- perplexite retenue par epoque (moyenne et ecart-type sur 3 graines)
 epoque    ppl (moy)    ppl (ect)
      1       27.111        0.426
      2       24.658        0.160
      3       24.447        0.144
      4       25.828        0.504
      5       26.560        0.224
      6       26.167        0.255
      7       26.392        0.404
      8       27.306        0.513
      9       27.919        0.509
     10       28.281        0.474
     11       28.457        0.588
     12       28.617        0.713
     13       28.821        0.809
     14       28.948        0.701
     15       28.990        0.659


### Lecture du résultat — le minimum est aux époques 2-3, pas à 15

La courbe moyenne dit la même chose que les trois graines prises une à une : la perplexité retenue chute pendant les deux premières époques (27,111 → 24,658), atteint son minimum à l'époque 3 (**24,447**, écart-type 0,144), puis **remonte sans retour** jusqu'à 28,990 à l'époque 15. Les douze dernières époques dégradent le modèle sur les textes qu'il n'a pas vus — pendant que la perte d'entraînement, elle, continue de descendre : c'est la définition même du sur-entraînement, et il coûte ici **4,5 points** de perplexité retenue, plus que la plupart des écarts d'hyperparamètres mesurés dans cette section.

Et la meilleure époque bouge d'une graine à l'autre (2, 3, 3) : même si le régime général est stable — minimum précoce, remontée ensuite — un budget fixe de quinze époques ne peut pas être optimal pour tous les tirages. La seule règle qui s'adapte au tirage est un critère d'arrêt : mesurer la perplexité retenue à chaque époque et s'arrêter quand elle ne s'améliore plus. C'est exactement ce que l'exercice 2 construit.

### Exercices à compléter

Quatre exercices non résolus ferment le notebook. Chacun mesure quelque chose que les exemples guidés ne mesurent pas : l'erreur commise sur les poids eux-mêmes, la généralisation (et non la mémorisation), le choix NF4 contre FP4, et l'échelle des mises à jour LoRA. Les stubs s'exécutent sans erreur ; complétez-les cellule par cellule.

#### Exercice 1 : erreur de reconstruction des poids

L'exemple guidé 1 a mesuré ce que la quantization coûte **en VRAM et en perplexité** — jamais ce qu'elle fait aux **poids eux-mêmes**. Ici on ouvre le capot : sur une seule couche `nn.Linear` d'OPT-1.3B (`model.decoder.layers[0].self_attn.q_proj`), comparer les poids FP16 à leurs versions NF4 et INT8 **reconstruites**, et localiser où l'erreur se concentre.

1. Charger OPT-1.3B en FP16 **sur CPU** (`device_map="cpu"`), extraire `W_fp16` (2048 × 2048) de `q_proj` de la couche 0, puis ne travailler qu'avec cette matrice.
2. Round trip NF4 : `bitsandbytes.functional.quantize_4bit` puis `dequantize_4bit` (l'API attend des blocs de 64 contigus, la matrice doit être reshapée en `(n_blocs, 64)` et être sur CUDA).
3. Round trip INT8 absmax **par ligne** : chaque ligne divisée par son max absolu, arrondie sur 127 niveaux, puis remultipliée.
4. Pour chaque format : erreur relative de Frobenius `‖W_rec − W‖_F / ‖W‖_F` et erreur absolue maximale.
5. Localiser l'erreur : trier les poids par magnitude absolue et calculer la part de l'erreur totale portée par le 1 % des poids les plus grands — les **outliers** que la quantization par blocs cherche à protéger.

**Indices** :
- La matrice seule fait 2048 × 2048 : la garder sur CPU et ne déplacer sur CUDA que ce que `bnb.functional` exige.
- Comparer aussi la forme des deux erreurs : NF4 est taillé pour une distribution de poids gaussienne, INT8 absmax pour une distribution uniforme.

**Critère de réussite** : un tableau `format | erreur Frobenius relative | erreur max absolue | part de l'erreur dans le 1 % des plus grands poids`, avec des erreurs relatives de l'ordre de 10⁻² à 10⁻¹.

In [30]:
# Exercice 1 : erreur de reconstruction des poids (NF4 / INT8 contre FP16)
# Etape 1 : charger OPT-1.3B sur CPU, extraire W_fp16 de q_proj (couche 0).
# Etape 2 : round trip NF4 (quantize_4bit/dequantize_4bit, blocs de 64) et INT8 absmax par ligne.
# Etape 3 : erreur relative de Frobenius + erreur absolue maximale pour chaque format.
# Etape 4 : part de l'erreur totale portee par le 1 % des poids de plus grande magnitude.
# Indice : bnb.functional travaille sur CUDA ; ne mover que la matrice 2048 x 2048.

W_fp16 = None      # TODO etudiant
W_rec_nf4 = None   # TODO etudiant
W_rec_int8 = None  # TODO etudiant

print("Exercice a completer : erreur de reconstruction des poids NF4/INT8 vs FP16")

Exercice a completer : erreur de reconstruction des poids NF4/INT8 vs FP16


#### Exercice 2 : arrêt précoce sur la perplexité retenue

La courbe de la section précédente montre **où** la perplexité retenue cesse de baisser, mais elle ne dit pas **quoi en faire** : le carnet entraîne encore quinze époques fixes et garde l'adaptateur de la dernière, quel que soit le trajet. Un entraînement réel ne paie pas la mesure d'un corpus retenu à chaque époque sans en tirer parti.

L'arrêt précoce est la règle qui exploite cette courbe : on garde l'adaptateur de la meilleure époque, et on arrête dès que la perplexité retenue n'a pas progressé depuis `patience` époques.

1. Écrire `entrainer_avec_arret_precoce(config, patience)`, sur le modèle de `entrainer_avec_courbe`, qui :
   - mesure la perplexité retenue à chaque fin d'époque (même sonde `SondeRetenue`) ;
   - **conserve une copie de l'état de l'adaptateur** à chaque nouvelle meilleure époque ;
   - **interrompt l'entraînement** (`control.should_training_stop = True`) dès que `patience` époques se sont écoulées sans amélioration.
2. Rejouer sur les trois graines `0`, `7` et `42`, avec `patience = 3`.
3. Reporter un tableau : `graine | meilleure époque | ppl retenue à l'arrêt | ppl retenue à l'époque 15 | époques économisées`.
4. Conclure : la meilleure époque est-elle la même pour les trois graines ? Si oui, le budget fixe de quinze époques est un mauvais réglage ; si non, l'arrêt précoce est la seule règle qui s'adapte au tirage.

**Indices** :

- La sonde voit le même objet modèle que l'entraînement : `{k: v.detach().clone() for k, v in modele.named_parameters() if v.requires_grad}` capture l'adaptateur seul, sans les poids quantifiés figés — c'est ce qui rend la copie peu coûteuse.
- `control.should_training_stop = True` dans `on_epoch_end` arrête proprement la boucle ; `state.global_step` reste cohérent avec les époques réellement effectuées.
- Comparer à **travail effectué**, pas seulement à perplexité : l'époque 15 coûte quinze époques, l'arrêt précoce en coûte `meilleure époque + patience`. Un gain de perplexité obtenu en deux fois moins d'époques est un gain réel ; un gain nul payé au même prix ne l'est pas.

**Critère de réussite** : le tableau des trois graines avec la meilleure époque, la perplexité retenue à l'arrêt et celle de l'époque 15, et une phrase qui dit si le gain dépasse la dispersion entre graines.

In [31]:
# Exercice 2 : arret precoce sur la perplexite retenue
# Etape 1 : ecrire entrainer_avec_arret_precoce(config, patience) sur le modele
#           de `entrainer_avec_courbe` : sonde a chaque epoque, copie des
#           parametres entrainables a chaque nouvelle meilleure epoque, et
#           interruption (control.should_training_stop = True) apres `patience`
#           epoques sans amelioration.
# Etape 2 : rejouer sur les graines 0, 7 et 42 avec patience = 3.
# Etape 3 : comparer la perplexite retenue a l'arret, celle de l'epoque 15,
#           et le nombre d'epoques reellement effectuees.
# Indice : seuls les parametres entrainables portent l'adaptateur (requires_grad),
#          et l'arret precoce coute `meilleure epoque + patience` epoques.

resultats_arret_precoce = None  # TODO etudiant

print("Exercice a completer : a quelle epoque arreter sur chaque graine, "
      "et que coute l'arret precoce face aux quinze epoques fixes ?")

Exercice a completer : a quelle epoque arreter sur chaque graine, et que coute l'arret precoce face aux quinze epoques fixes ?


#### Exercice 3 : NF4 contre FP4 à mémoire égale

NF4 et FP4 occupent exactement la même place (4 bits par poids) : la différence est purement dans la **forme des niveaux de quantization**. Comparer les deux à mémoire égale sur OPT-1.3B.

1. Charger OPT-1.3B deux fois en 4-bit : `bnb_4bit_quant_type="nf4"` puis `"fp4"` — la double quantization reste identique dans les deux configurations.
2. Mesurer l'empreinte (`get_memory_footprint`) et le pic de chargement pour chaque format : ils doivent être quasi identiques.
3. Mesurer la perplexité sur les textes d'entraînement pour les deux formats.
4. Générer en greedy sur un même prompt et comparer les sorties.

**Indices** :
- Les poids pré-entraînés d'un transformer suivent une distribution à peu près **normale**, centrée et concentrée près de zéro : NF4 place ses niveaux sur les quantiles de cette distribution (plus de niveaux au centre). **FP4 n'est pas une grille uniforme** : c'est un format flottant E2M1 (1 bit de signe, 2 d'exposant, 1 de mantisse) dont les niveaux positifs normalisés, tels qu'implémentés dans bitsandbytes (`functional.py`, SHA `833649043474794b8fe7a4136e0c40faf077b2e0`), sont `0, 0.0052, 0.1667, 0.25, 0.3333, 0.5, 0.6667, 1` — les écarts ne sont pas constants (densité plus grande au centre qu'aux extrêmes, mais moins extrême que NF4). NF4 *devrait* donc gagner en perplexité à mémoire égale sur des poids gaussiens, mais c'est une prédiction, pas un résultat garanti.
- Reprendre le squelette de l'exemple guidé 3 en ne changeant QUE `bnb_4bit_quant_type`.

**Critère de réussite** : empreintes égales à ~1 Mo près, les deux générations côte à côte, et un tableau ppl(NF4) vs ppl(FP4). Toute direction de l'inégalité (NF4 < FP4, NF4 ≈ FP4, NF4 > FP4) est acceptable si le protocole et les mesures la soutiennent — le classement est un résultat expérimental, pas un attendu.

In [32]:
# Exercice 3 : NF4 contre FP4 à mémoire égale
# Etape 1 : charger OPT-1.3B en 4-bit avec bnb_4bit_quant_type="nf4" puis "fp4"
#           (bnb_4bit_use_double_quant identique dans les deux).
# Etape 2 : empreinte (get_memory_footprint) et pic de chargement pour chaque format.
# Etape 3 : perplexite sur les textes d'entrainement + generation greedy sur un meme prompt.
# Indice : ne changer QUE bnb_4bit_quant_type entre les deux configurations.

resultats_exo3 = None  # TODO etudiant

print("Exercice a completer : NF4 vs FP4 a memoire egale")

Exercice a completer : NF4 vs FP4 a memoire egale


#### Exercice 4 : échelle alpha/r contre alpha/sqrt(r) (rsLoRA)

Le `LoRALinear` de la section 4 met la mise à jour à l'échelle par `alpha / r`. rsLoRA propose `alpha / sqrt(r)` : à rang élevé, la mise à jour ne s'effondre pas. Testez-le sur la régression jouet de la section 4 — CPU, rapide, sans modèle lourd.

1. Reprendre la classe `LoRALinear` de la section 4 (copie locale avec un paramètre d'échelle), sur la même tâche de régression : même couche de base, mêmes données, graines fixées.
2. Avec `alpha` fixé (par ex. 16), entraîner r = 4, 16, 64 pendant le même nombre de pas SGD (par ex. 300), même learning rate, pour les deux échelles : `alpha/r` et `alpha/sqrt(r)`.
3. Reporter la perte finale des 6 configurations dans un tableau.

**Indices** :
- À `alpha` fixé, l'échelle `alpha/r` décroît comme 1/r : doubler le rang divise la mise à jour par deux ; `alpha/sqrt(r)` ne la divise que par √2. C'est l'hypothèse à confronter à la mesure.
- Fixer toutes les graines (`torch.manual_seed`) pour que la seule différence entre deux runs soit l'échelle.

**Critère de réussite** : à r = 64, une perte finale nettement plus basse avec `alpha/sqrt(r)` qu'avec `alpha/r`, et des pertes proches des deux échelles à r = 4.

In [33]:
# Exercice 4 : échelle alpha/r contre alpha/sqrt(r) (rsLoRA) sur la régression de la section 4
# Etape 1 : reprendre LoRALinear avec un parametre d'echelle, alpha fixe (ex. 16), CPU.
# Etape 2 : entrainer r = 4, 16, 64 sur le meme nombre de pas SGD pour les deux echelles.
# Etape 3 : tableau des pertes finales des 6 configurations.
# Indice : graines fixees, seule l'echelle doit differer entre deux runs.

resultats_exo4 = None  # TODO etudiant

print("Exercice a completer : alpha/r vs alpha/sqrt(r) a r = 4, 16, 64")

Exercice a completer : alpha/r vs alpha/sqrt(r) a r = 4, 16, 64


## Resume

| Aspect | LoRA (FT-01) | QLoRA (FT-02) |
|--------|-------------|---------------|
| Poids du modèle | FP16 (16 bits) | NF4 (4 bits) |
| Adaptateurs | LoRA en FP16 | LoRA en FP16 |
| Calculs forward | FP16 natif | Dequantize → FP16 → re-quantize |
| Économie mémoire | ~1.1x modèle base | **~4x sur les poids seuls** (0,41 contre 1,63 Go) |
| Qualité | ~98-99% du full FT | ~97-99% du full FT *(ordre de grandeur)* |
| Modèle recommandé | < 1B params | **> 1B params** |

**Points clés** :
- **QLoRA = Quantization (NF4) + LoRA** : le meilleur des deux mondes
- **NF4** est optimal pour les poids (distribution normale), pas un simple INT4
- **Double quantization** compresse les scaling factors pour des gains supplémentaires
- **`paged_adamw_8bit`** : optimiseur en 8-bit pour économiser encore plus de VRAM
- Un modèle 7B en QLoRA tient sur un GPU 6 Go — démocratise le fine-tuning

**Référence** : Dettmers et al., "QLoRA: Efficient Finetuning of Quantized LLMs" (2023).

**Prochaines étapes** (FT-03) : Fine-tuning multi-tâches avec plusieurs adaptateurs LoRA.


**Le parcours mesuré se lit dans les outputs** : le chargement NF4, le nombre exact de
paramètres entraînables, la durée, le pic VRAM et la perte finale sont produits par les
cellules précédentes. La grille de la section 4 prédit le compte d'adaptateurs au paramètre
près ; les générations de la section 7 montrent ensuite le mémo-styling à l'œuvre. Le
balayage étendu de la section 10 ajoute 30 entraînements sur trois graines : le rang et le
rapport alpha/r déplacent la perplexité retenue de plusieurs points, les modules ciblés
n'en déplacent pas assez pour sortir du bruit, et la courbe par époque situe le minimum aux
époques 2-3. La boucle FT-01 → FT-02 est close : même mécanique LoRA, seule la résidence
mémoire des poids change. FT-03 ouvre le multi-tâches : plusieurs adaptateurs, un modèle de
base, et la question du partage de rang.


**Les deux lignes du tableau qui ne sont pas des mesures** : la ligne « Qualité » et le
chiffre « ~3-4x » de l'économie mémoire sont des **ordres de grandeur** (rapport de poids
théoriques et papier QLoRA), pas des sorties de ce carnet. Ce que l'exécution **mesure** :
l'empreinte résiduelle du modèle 4 bits après chargement (**2,72 Go**) contre sa théorie de
poids seuls (**0,41 Go**), le pic de chargement du même modèle (**8,06 Go**, au-dessus du
FP16 à 7,39 Go -- le transitoire de conversion domine ce pic), le pic d'entraînement QLoRA
de la configuration nominale (**2,13 Go**), et la perplexité sur le corpus retenu
(**29,440** après adaptateur contre **31,594** pour la base NF4 seule). C'est sur ces
nombres -- et non sur les ordres de grandeur du tableau -- que porte la lecture du carnet.


**Vérification de l'affirmation « un 7B tient sur 6 Go »** : poids 4-bit = 0,5 octet par
paramètre, soit 3,5 Go pour 7 Md, contre 14 Go en FP16. Les adaptateurs LoRA et
l'optimiseur paginé portent sur une petite fraction des paramètres. La marge dépend ensuite
de la longueur de contexte, du batch et des activations : c'est pourquoi le notebook mesure
le pic réel au lieu de se contenter de l'arithmétique des poids.